## Система запуска

- **Разработка и обучение:** Windows, VS Code, Python 3.12, GPU NVIDIA RTX 5070 (12 ГБ).
- **Финальный инференс и проверка воспроизводимости:** Linux (Google Colab, Python 3.13). Для полного расчёта без кэшей нужен GPU с BF16 и достаточной памятью (L4 или A100; T4 не подходит).
- **Файл `submission.csv`** сформирован в Google Colab.
- Вне Colab (VS Code, Windows/Linux) первая ячейка ставит пакеты из `requirements.txt`; в Colab ставятся собственные версии пакетов, torch берётся предустановленный.


In [1]:
import os, sys, subprocess, platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Система:", platform.system(), platform.release(), "| Python", platform.python_version(), "| Colab:", IN_COLAB)

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *args])

if os.getenv("RANK_SKIP_INSTALL"):
    print("Установка пакетов пропущена (RANK_SKIP_INSTALL).")
elif IN_COLAB:
    _pip("numpy>=1.26,<3", "pandas>=2.0,<3", "transformers==4.57.1", "huggingface_hub>=0.34,<1.0",
         "safetensors>=0.6", "lightgbm==4.6.0", "filelock>=3.13", "scikit-learn>=1.5",
         "threadpoolctl>=3.5", "sentencepiece>=0.2", "accelerate==1.10.1", "bitsandbytes==0.48.1")
else:
    requirements = Path.cwd() / "requirements.txt"
    if requirements.is_file():
        _pip("-r", str(requirements))
    else:
        print("requirements.txt не найден в", Path.cwd(), "- ставлю минимальный набор.")
        _pip("numpy>=1.26,<3", "pandas>=2.0,<3", "transformers==4.57.1", "huggingface_hub>=0.34,<1.0",
             "safetensors>=0.6", "lightgbm==4.6.0", "filelock>=3.13", "scikit-learn>=1.5",
             "threadpoolctl>=3.5", "sentencepiece>=0.2", "accelerate==1.10.1", "bitsandbytes==0.48.1")

import torch

# --- Необязательные кэши измерений ------------------------------------------
cache_root = Path("/content") if IN_COLAB else Path.cwd()
cache_paths = {
    "RANK_REPLAY": cache_root / "RANK_REPLAY" / "base-features.npz",
    "RANK_JINA_FULL_LIST": cache_root / "RANK_JINA_FULL_LIST" / "scores.npz",
    "RANK_JINA_REVERSE_FULL": cache_root / "RANK_JINA_REVERSE_FULL" / "scores.npz",
}
for var, path in cache_paths.items():
    if path.is_file():
        os.environ[var] = str(path)
        print(f"🟢 Найден: {var}")
    else:
        os.environ.pop(var, None)
        print(f"⚪ Не найден: {var}")

if all(os.getenv(var) for var in cache_paths):
    print("\nРЕЖИМ: все кэши найдены, используем их.")
else:
    print("\nРЕЖИМ: не все кэши найдены. Недостающие признаки рассчитываются.")

_runtime_notebook = Path.cwd() / "solution_selfcontained_runtime.ipynb"
if not _runtime_notebook.exists():
    _runtime_notebook.write_text("Self-contained ranking notebook runtime marker.", encoding="utf-8")
os.environ.setdefault("RANK_NOTEBOOK", str(_runtime_notebook))

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
REPLAY_READY = all(os.getenv(name) for name in cache_paths)
if REPLAY_READY:
    print("Режим воспроизведения: все три проверенных кэша найдены; допускается CPU.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Для расчёта признаков без полного replay-кэша нужен CUDA GPU. В Colab выбери GPU.")
    print("GPU:", torch.cuda.get_device_name(0),
          "| видеопамять, ГБ:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    bf16_supported = torch.cuda.is_bf16_supported()
    print("BF16 поддерживается:", bf16_supported)
    if not bf16_supported:
        raise RuntimeError("Для расчёта без полного replay-кэша нужен GPU с поддержкой BF16 (например, L4/A100).")
print("\nПроверка первой ячейки завершена.")


Система: Linux 6.6.159+ | Python 3.13.16 | Colab: True
🟢 Найден: RANK_REPLAY
🟢 Найден: RANK_JINA_FULL_LIST
🟢 Найден: RANK_JINA_REVERSE_FULL

РЕЖИМ: все кэши найдены, используем их.
PyTorch: 2.14.1+cu130
CUDA доступна: True
Режим воспроизведения: все три проверенных кэша найдены; допускается CPU.

Проверка первой ячейки завершена.


## Данные, разбиение по запросам и NDCG@10

In [2]:
"""Data preparation and ranking validation used by the final ranking notebook.

Only the supplied competition CSV files are used. Corpus-dependent title
recovery is fitted on the training part of each fold, never on its validation
part. The metric averages exact score ties; the organizer's scorer should be
checked if a different tie convention is published.
"""

import hashlib
import html
import io
import json
import re
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


DATA_SCHEMA_VERSION = "ranking-data-v1"
PREPROCESS_VERSION = "ranking-text-v1"
BASE_COLUMNS = [
    "id", "query_id", "document_id", "article_id", "query", "title", "text",
    "passage_index", "article_passage_count", "article_char_count", "ingest_source",
]
FILE_COLUMNS = {
    "train.csv": BASE_COLUMNS + ["relevance"],
    "test.csv": BASE_COLUMNS,
    "sample_submission.csv": ["id", "prediction"],
}
_MOJIBAKE = re.compile(r"[РС][\u0080-\u00bf\u0400-\u04ff]")
_RUSSIAN_WORD = re.compile(r"[а-яё]{4,}", re.I)
_HTML_TAG = re.compile(r"<\s*/?\s*[A-Za-z][^>]*>")
_INVISIBLE = re.compile(r"[\u00ad\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]")
_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WIKIPEDIA_SUFFIX = re.compile(r"\s+[-–—]\s*Википедия\s*$", re.I)
_TITLE_PLACEHOLDERS = {"не указано", "не указан", "не указана"}


def fix_mojibake(value):
    """Conservatively repair reversible UTF-8 decoded as CP1251.

    Never use errors='ignore' or errors='replace': unrepairable text remains
    intact instead of silently losing characters. At most two reversible
    rounds are attempted, and a candidate must reduce the corruption signal.
    """
    if value is None or not isinstance(value, str):
        return "" if value is None or pd.isna(value) else str(value)
    text = value
    for _ in range(2):
        before = len(_MOJIBAKE.findall(text))
        if before <= 5:
            break
        try:
            candidate = text.encode("cp1251", errors="strict").decode("utf-8", errors="strict")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if "\ufffd" in candidate or not _RUSSIAN_WORD.search(candidate):
            break
        if len(_MOJIBAKE.findall(candidate)) >= before:
            break
        text = candidate
    return text


def clean_text(value):
    """Shared case-preserving cleaner for queries, titles, and passages."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    text = fix_mojibake(str(value))
    text = unicodedata.normalize("NFC", html.unescape(text))
    text = _HTML_TAG.sub(" ", text)
    text = _INVISIBLE.sub("", text)
    text = _CONTROL.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_title(value):
    text = _WIKIPEDIA_SUFFIX.sub("", clean_text(value)).strip()
    return "" if text.casefold() in _TITLE_PLACEHOLDERS else text


def _required_strings(frame, columns, filename):
    for column in columns:
        if frame[column].isna().any() or frame[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"{filename}: empty required field {column!r}")


def _read_csv_bytes(payload, filename):
    # Literal strings such as 'NA', corrupted numbers, and whitespace are data.
    frame = pd.read_csv(io.BytesIO(payload), encoding="utf-8-sig", dtype=str,
                        keep_default_na=False, na_filter=False)
    expected = FILE_COLUMNS[filename]
    if list(frame.columns) != expected:
        raise ValueError(f"{filename}: expected columns {expected}, got {list(frame.columns)}")
    if frame.empty:
        raise ValueError(f"{filename}: file contains no data rows")
    _required_strings(frame, ["id"], filename)
    if frame["id"].duplicated().any():
        raise ValueError(f"{filename}: duplicate id values")
    if filename != "sample_submission.csv":
        _required_strings(frame, ["query_id", "document_id", "article_id", "query", "text"], filename)
        if frame.duplicated(["query_id", "document_id"]).any():
            raise ValueError(f"{filename}: duplicate (query_id, document_id) pairs")
        normalized_queries = frame["query"].map(clean_text).str.casefold()
        if normalized_queries.eq("").any():
            raise ValueError(f"{filename}: query becomes empty after cleaning")
        if normalized_queries.groupby(frame["query_id"]).nunique().gt(1).any():
            raise ValueError(f"{filename}: one query_id has different query texts")
        if frame.groupby("document_id")["article_id"].nunique().gt(1).any():
            raise ValueError(f"{filename}: one document_id belongs to different articles")
    if filename == "train.csv":
        labels = pd.to_numeric(frame["relevance"], errors="raise")
        if not np.isfinite(labels.to_numpy(dtype=float)).all() or not labels.isin([0, 1]).all():
            raise ValueError("train.csv: relevance must contain only binary labels 0 and 1")
        frame["relevance"] = labels.astype(np.int8)
        classes = frame.groupby("query_id")["relevance"].agg(["min", "max"])
        if not (classes["min"].eq(0) & classes["max"].eq(1)).all():
            raise ValueError("train.csv: pairwise training requires positive and negative candidates for every query")
    return frame


def load_competition_data(path):
    """Read an explicit directory or ZIP containing the three exact CSV names.

    The ZIP may contain one common enclosing directory. Ambiguous duplicate
    filenames are rejected. No archive extraction or recursive file search is
    performed. Raw metadata values and empty strings are preserved.
    """
    path = Path(path).expanduser().resolve()
    payloads = {}
    if path.is_dir():
        for filename in FILE_COLUMNS:
            candidate = path / filename
            if not candidate.is_file():
                raise FileNotFoundError(f"Missing {candidate}. Set DATA_PATH to the competition directory or ZIP.")
            payloads[filename] = candidate.read_bytes()
    elif path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            members = [member for member in archive.infolist() if not member.is_dir()]
            for filename in FILE_COLUMNS:
                matches = [member for member in members if Path(member.filename).name == filename]
                if len(matches) != 1:
                    raise ValueError(f"ZIP must contain exactly one {filename}; found {len(matches)}")
                if matches[0].file_size > 512 * 1024 * 1024:
                    raise ValueError(f"Unexpectedly large CSV: {filename}")
                payloads[filename] = archive.read(matches[0])
    else:
        raise FileNotFoundError(f"DATA_PATH must be an existing competition directory or ZIP: {path}")

    frames = {name: _read_csv_bytes(payloads[name], name) for name in FILE_COLUMNS}
    train, test, sample = (frames[name] for name in FILE_COLUMNS)
    for column in ["id", "query_id", "document_id"]:
        if set(train[column]).intersection(test[column]):
            raise ValueError(f"Train/test overlap in {column}; review the dataset split before training")
    train_queries = set(train["query"].map(clean_text).str.casefold())
    test_queries = set(test["query"].map(clean_text).str.casefold())
    if train_queries.intersection(test_queries):
        raise ValueError("Train/test contain identical normalized query text under different IDs")
    train_passages = {hashlib.sha256(clean_text(text).encode("utf-8")).hexdigest() for text in train["text"]}
    test_passages = {hashlib.sha256(clean_text(text).encode("utf-8")).hexdigest() for text in test["text"]}
    if train_passages.intersection(test_passages):
        raise ValueError("Train/test contain identical cleaned passage text under different document IDs")
    if len(sample) != len(test) or set(sample["id"]) != set(test["id"]):
        raise ValueError("sample_submission.csv IDs must match test.csv exactly")
    files = {
        name: {"sha256": hashlib.sha256(payloads[name]).hexdigest(),
               "size_bytes": len(payloads[name]), "rows": len(frames[name]),
               "columns": list(frames[name].columns)}
        for name in FILE_COLUMNS
    }
    dataset_signature = hashlib.sha256(json.dumps(
        {name: files[name]["sha256"] for name in sorted(files)}, sort_keys=True
    ).encode("utf-8")).hexdigest()
    manifest = {"schema_version": DATA_SCHEMA_VERSION, "source": str(path),
                "dataset_sha256": dataset_signature, "files": files,
                "train_queries": int(train["query_id"].nunique()),
                "test_queries": int(test["query_id"].nunique()),
                "positive_rows": int(train["relevance"].sum()),
                "shared_articles": len(set(train["article_id"]).intersection(test["article_id"]))}
    return train, test, sample, manifest


def fit_title_map(frame):
    """Recover titles only when the training observations agree on a title."""
    candidates = pd.DataFrame({"article_id": frame["article_id"].astype(str).to_numpy(),
                               "title": frame["title"].map(clean_title).to_numpy()})
    candidates = candidates[candidates["title"].ne("")]
    result = {}
    for article_id, values in candidates.groupby("article_id", sort=True)["title"]:
        distinct = sorted(set(values), key=lambda value: (value.casefold(), value))
        if len({value.casefold() for value in distinct}) == 1:
            result[article_id] = distinct[0]
    return result


def prepare_frame(frame, title_map):
    """Retain source columns and add cleaned text using a train-fitted map."""
    prepared = frame.copy().reset_index(drop=True)
    prepared["query_clean"] = prepared["query"].map(clean_text)
    prepared["title_clean"] = prepared["title"].map(clean_title)
    prepared["text_clean"] = prepared["text"].map(clean_text)
    missing = prepared["title_clean"].eq("")
    prepared["title_was_missing"] = missing.astype(np.int8)
    recovered = prepared["article_id"].astype(str).map(title_map).fillna("")
    prepared.loc[missing, "title_clean"] = recovered.loc[missing]
    prepared["title_recovered"] = (missing & prepared["title_clean"].ne("")).astype(np.int8)
    if prepared["query_clean"].eq("").any():
        raise ValueError("A query is empty after preprocessing")
    if prepared["text_clean"].eq("").any():
        raise ValueError("A passage is empty after preprocessing; inspect the original row")
    return prepared


def build_group_folds(train, n_folds=5):
    """Group connected queries/documents and identical cleaned text.

    Component IDs are full SHA-256 digests of their sorted members, so grouping
    does not depend on Python's salted hash or source row order.
    """
    if not isinstance(n_folds, int) or isinstance(n_folds, bool) or n_folds < 2:
        raise ValueError("n_folds must be an integer >= 2")
    parent = {}

    def find(node):
        parent.setdefault(node, node)
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(first, second):
        first, second = find(first), find(second)
        if first != second:
            lo, hi = sorted((first, second))
            parent[hi] = lo

    normalized_queries = train["query"].map(clean_text).str.casefold().to_numpy()
    passage_hashes = np.array([hashlib.sha256(clean_text(text).encode("utf-8")).hexdigest()
                               for text in train["text"]])
    for qid, did, text, passage_hash in zip(train["query_id"], train["document_id"],
                                            normalized_queries, passage_hashes):
        union(("query_id", str(qid)), ("document_id", str(did)))
        union(("query_id", str(qid)), ("query_text", str(text)))
        union(("document_id", str(did)), ("passage_text_sha256", str(passage_hash)))
    members = {}
    for node in list(parent):
        members.setdefault(find(node), []).append(node)
    component_ids = {
        root: hashlib.sha256(json.dumps(sorted(nodes), ensure_ascii=False,
                                       separators=(",", ":")).encode("utf-8")).hexdigest()
        for root, nodes in members.items()
    }
    groups = np.array([component_ids[find(("query_id", str(qid)))]
                       for qid in train["query_id"]], dtype=str)
    if len(set(groups)) < n_folds:
        raise ValueError(f"Only {len(set(groups))} independent components for {n_folds} folds")
    splitter = GroupKFold(n_splits=n_folds)
    folds = list(splitter.split(np.zeros(len(train)), train["relevance"], groups))
    covered = np.zeros(len(train), dtype=np.int8)
    for training, validation in folds:
        for column in ["query_id", "document_id"]:
            if set(train.iloc[training][column]).intersection(train.iloc[validation][column]):
                raise AssertionError(f"Fold leakage in {column}")
        if set(normalized_queries[training]).intersection(normalized_queries[validation]):
            raise AssertionError("Fold leakage in normalized query text")
        if set(passage_hashes[training]).intersection(passage_hashes[validation]):
            raise AssertionError("Fold leakage in cleaned passage text")
        if set(groups[training]).intersection(groups[validation]):
            raise AssertionError("Fold leakage in connected components")
        covered[validation] += 1
    if not np.all(covered == 1):
        raise AssertionError("Each row must occur in exactly one validation fold")
    return folds, groups


def fold_summary(train, folds, groups):
    rows = []
    for fold, (training, validation) in enumerate(folds, 1):
        rows.append({"fold": fold, "train_rows": len(training),
                     "validation_rows": len(validation),
                     "train_queries": int(train.iloc[training]["query_id"].nunique()),
                     "validation_queries": int(train.iloc[validation]["query_id"].nunique()),
                     "validation_components": len(set(groups[validation])),
                     "validation_positives": int(train.iloc[validation]["relevance"].sum())})
    return pd.DataFrame(rows)


def _checked_scores(pred, length):
    scores = np.asarray(pred, dtype=np.float64)
    if scores.shape != (length,):
        raise ValueError(f"Expected {length} scores, got shape {scores.shape}")
    if not np.isfinite(scores).all():
        raise ValueError("Predictions contain NaN or infinity")
    return scores


def ndcg_per_query(df, pred, k=10):
    """Binary NDCG@k using the organizer's exact tie-break rule.

    Sort by descending prediction and then by id ascending as a string.
    Queries without positive labels receive 0.
    """
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError("k must be a positive integer")
    scores = _checked_scores(pred, len(df))
    if df.empty or df["query_id"].isna().any():
        raise ValueError("Metric requires nonempty data and nonmissing query IDs")
    if "id" not in df.columns or df["id"].isna().any():
        raise ValueError("Metric requires nonmissing candidate ids for official tie-breaking")
    relevance = pd.to_numeric(df["relevance"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(relevance).all() or not np.isin(relevance, [0, 1]).all():
        raise ValueError("Metric expects finite binary relevance labels")
    ids = df["id"].astype(str).to_numpy()
    result = {}
    query_positions = pd.Series(np.arange(len(df))).groupby(
        df["query_id"].to_numpy(), sort=True).indices
    for query_id, positions in query_positions.items():
        rel, score = relevance[positions], scores[positions]
        discounts = 1.0 / np.log2(np.arange(2, min(k, len(rel)) + 2))
        ideal = float(np.dot(np.sort(rel)[::-1][:len(discounts)], discounts))
        if ideal == 0.0:
            result[query_id] = 0.0
            continue
        order = np.lexsort((ids[positions], -score))
        dcg = float(np.dot(rel[order][:len(discounts)], discounts))
        result[query_id] = dcg / ideal
    return pd.Series(result, name=f"ndcg@{k}").rename_axis("query_id")

def mean_ndcg(df, pred, k=10):
    return float(ndcg_per_query(df, pred, k=k).mean())


def normalize_by_query(pred, qids):
    """Population z-score per query, constant/singleton groups mapped to zero."""
    qids = np.asarray(qids)
    if qids.ndim != 1 or pd.isna(qids).any():
        raise ValueError("Expected a one-dimensional array of nonmissing query IDs")
    scores = _checked_scores(pred, len(qids))
    series = pd.Series(scores)
    grouped = series.groupby(qids, sort=False)
    means = grouped.transform("mean").to_numpy()
    std = grouped.transform("std", ddof=0).to_numpy()
    normalized = np.zeros_like(scores)
    variable = std > 1e-12
    normalized[variable] = (scores[variable] - means[variable]) / std[variable]
    if not np.isfinite(normalized).all():
        raise ValueError("Nonfinite scores after within-query normalization")
    return normalized


def build_submission(sample, test, pred):
    """Align predictions by id to sample order and validate every output row."""
    scores = _checked_scores(pred, len(test))
    if sample["id"].isna().any() or test["id"].isna().any():
        raise ValueError("Submission IDs cannot be missing")
    if sample["id"].duplicated().any() or test["id"].duplicated().any():
        raise ValueError("Submission IDs must be unique")
    if len(sample) != len(test) or set(sample["id"]) != set(test["id"]):
        raise ValueError("Sample/test ID sets differ")
    keyed = pd.Series(scores, index=test["id"].to_numpy())
    result = pd.DataFrame({"id": sample["id"].to_numpy(copy=True),
                           "prediction": keyed.loc[sample["id"].to_numpy()].to_numpy()})
    if list(result.columns) != ["id", "prediction"] or not np.isfinite(result["prediction"]).all():
        raise AssertionError("Invalid final submission")
    return result

## Дообучение BGE по группам и контрольные точки

In [3]:
"""Grouped BGE trainer. Importing/defining this component performs no model or data work."""
import hashlib
import json
import math
import os
import random
import time
from functools import lru_cache
from importlib import metadata
from pathlib import Path
from uuid import uuid4

import numpy as np
import torch
from filelock import FileLock
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


BGE_NIGHT_CONFIG = {
    "model": "BAAI/bge-reranker-v2-m3", "revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
    "seed": 322, "epochs": 2, "max_length": 384, "top_layers": 12,
    "query_batch": 8, "sequence_microbatch": 4, "predict_batch": 24, "prediction_chunk": 384,
    "learning_rate": 1e-5, "weight_decay": 0.01, "warmup_ratio": 0.05, "max_grad_norm": 1.0,
    "checkpoint_steps": 50, "precision": "bf16_if_supported",
}
_BGE_FILES = {
    "model.safetensors": "d9e3e081faff1eefb84019509b2f5558fd74c1a05a2c7db22f74174fcedb5286",
    "config.json": "9f62673cb00ec41dcec8947b9ed16f6f2eb23ba2",
    "tokenizer.json": "69564b696052886ed0ac63fa393e928384e0f8caada38c1f4864a9bfbf379c15",
    "tokenizer_config.json": "328a00a9a560aadcf2a3064f917517359eb3cc26",
    "special_tokens_map.json": "b1879d702821e753ffe4245048eee415d54a9385",
    "sentencepiece.bpe.model": "cfc8146abe2a0488e9e2a0c56de7952f7c11ab059eca145a0a727afce0db2865",
}


@lru_cache(maxsize=32)
def _bge_verified_file(path, size, mtime_ns, expected):
    digest = hashlib.sha256() if len(expected) == 64 else hashlib.sha1(f"blob {size}\0".encode())
    with open(path, "rb") as source:
        for content in iter(lambda: source.read(4 << 20), b""):
            digest.update(content)
    if digest.hexdigest() != expected:
        raise ValueError(f"{path} differs from the pinned BGE revision.")
    return expected


def _bge_frame_hash(frame, labeled):
    columns = ["id", "query_id", "query_clean", "title_clean", "text_clean"] + (["relevance"] if labeled else [])
    if frame.empty or not frame.id.is_unique or frame[columns].isna().any().any():
        raise ValueError("Expected nonempty cleaned data with unique IDs and nonmissing fields.")
    digest = hashlib.sha256()
    for row in frame[columns].itertuples(index=False, name=None):
        for field in row:
            encoded = str(field).encode("utf-8")
            digest.update(len(encoded).to_bytes(8, "little"))
            digest.update(encoded)
    return digest.hexdigest()


def _bge_atomic(path, writer):
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    try:
        with temp.open("wb") as stream:
            writer(stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def _bge_report(path, value):
    data = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8")
    _bge_atomic(path, lambda stream: stream.write(data))


def _bge_encode(frame, tokenizer, max_length):
    result = []
    for begin in range(0, len(frame), 256):
        block = frame.iloc[begin:begin + 256]
        docs = [(title + " " + text) if title else text for title, text in zip(block.title_clean, block.text_clean)]
        packed = tokenizer(block.query_clean.tolist(), docs, padding=False, truncation="only_second", max_length=max_length)
        result.extend(packed["input_ids"])
    return result


def _bge_pad(rows, pad_id, device):
    width = 8 * math.ceil(max(map(len, rows)) / 8)
    ids = np.full((len(rows), width), pad_id, dtype=np.int64)
    mask = np.zeros_like(ids)
    for i, row in enumerate(rows):
        n = len(row)
        ids[i, :n] = row
        mask[i, :n] = 1
    return torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)


def _bge_model(model_dir, cfg, device, dtype):
    model, info = AutoModelForSequenceClassification.from_pretrained(
        model_dir, local_files_only=True, trust_remote_code=False, use_safetensors=True, output_loading_info=True)
    if (model.config.model_type != "xlm-roberta" or model.config.num_hidden_layers != 24
            or model.config.num_labels != 1 or info["mismatched_keys"]
            or set(info["missing_keys"]).intersection(dict(model.named_parameters()))):
        raise ValueError("A complete pretrained pinned single-score XLM-R BGE model is required.")
    model.train()
    model.requires_grad_(False)
    layers = model.roberta.encoder.layer
    for part in (model.roberta.embeddings, *layers[:-cfg["top_layers"]]):
        part.eval()
        part.to(dtype=dtype)
    for part in (*layers[-cfg["top_layers"]:], model.classifier):
        part.float()
        part.requires_grad_(True)
    return model.to(device)


def _bge_query_loss(model, tokens, positions, labels, cfg, pad_id, device, dtype):
    # Length ordering is within the query. Restore labels in the same order.
    positions = sorted(positions, key=lambda i: len(tokens[i]))
    target = torch.as_tensor(labels[positions], device=device, dtype=torch.bool)
    micro = cfg["sequence_microbatch"]
    def forward(ids, mask):
        with torch.autocast(device.type, dtype=dtype, enabled=device.type == "cuda"):
            return model(input_ids=ids, attention_mask=mask).logits.float().reshape(-1)
    parts = []
    for begin in range(0, len(positions), micro):
        ids, mask = _bge_pad([tokens[i] for i in positions[begin:begin + micro]], pad_id, device)
        # Only oversized queries need activation recomputation. Nonreentrant
        # checkpointing supports integer inputs; CUDA RNG is restored for dropout.
        if len(positions) > micro:
            parts.append(checkpoint(forward, ids, mask, use_reentrant=False, preserve_rng_state=True))
        else:
            parts.append(forward(ids, mask))
    scores = torch.cat(parts)
    positive, negative = scores[target], scores[~target]
    if not positive.numel() or not negative.numel():
        raise ValueError("A training query must contain both relevance labels.")
    # Every positive-negative comparison gets the same weight within this query.
    loss = sum(F.softplus(chunk[:, None] - positive[None, :]).sum() for chunk in negative.split(32))
    return loss / (positive.numel() * negative.numel())


def _bge_rng(device):
    n = np.random.get_state()
    return {"python": random.getstate(), "numpy": (n[0], n[1].tolist(), n[2], n[3], n[4]),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def _bge_restore_rng(state, device):
    random.setstate(state["python"])
    n = state["numpy"]
    np.random.set_state((n[0], np.array(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(state["torch"])
    if device.type == "cuda":
        torch.cuda.set_rng_state(state["cuda"], device)


def _bge_weights(model):
    return {name: p.detach().cpu() for name, p in model.named_parameters() if p.requires_grad}


def _bge_weight_id(weights):
    digest = hashlib.sha256()
    for name in sorted(weights):
        digest.update(name.encode("utf-8") + b"\0")
        digest.update(memoryview(weights[name].contiguous().numpy()).cast("B"))
    return digest.hexdigest()


def _bge_restore_weights(model, weights):
    names = {name for name, p in model.named_parameters() if p.requires_grad}
    if set(weights) != names:
        raise ValueError("Checkpoint trainable parameter names differ from the model.")
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in weights:
                if p.shape != weights[name].shape:
                    raise ValueError(f"Checkpoint parameter shape differs for {name}.")
                p.copy_(weights[name])


def _bge_train(model, frame, tok, cfg, fold, device, dtype, signature, path, stop_requested, progress):
    groups = list(frame.groupby("query_id", sort=True).indices.values())
    labels = frame.relevance.to_numpy(dtype=float)
    if not np.isin(labels, [0., 1.]).all() or any(np.unique(labels[g]).size != 2 for g in groups):
        raise ValueError("Training requires binary labels and both classes for every query.")
    group_batch = cfg["query_batch"]
    per_epoch = math.ceil(len(groups) / group_batch)
    total = cfg["epochs"] * per_epoch
    cursor = 0
    applied = 0
    train_seconds = 0.0
    saved = None
    if path.is_file():
        saved = torch.load(path, map_location="cpu", weights_only=True)
        if saved["signature"] != signature or saved["total"] != total:
            raise ValueError("Training checkpoint signature or schedule mismatch.")
        cursor, applied, train_seconds = saved["cursor"], saved["applied"], saved["train_seconds"]
        if not (0 <= applied <= cursor <= total):
            raise ValueError("Invalid checkpoint cursor.")
        weights = saved.pop("weights")
        _bge_restore_weights(model, weights)
        if saved["complete"]:
            if cursor != total or applied != total or _bge_weight_id(weights) != saved["weight_id"]:
                raise ValueError("Incomplete training marked complete.")
            _bge_report(path.parent / "trained.json", {"signature": signature, "weight_id": saved["weight_id"]})
            return True, {"train_steps": applied, "train_seconds": train_seconds, "weight_id": saved["weight_id"]}
        del weights
    if stop_requested():
        return False, {"train_steps": applied, "train_seconds": train_seconds}
    tokens = _bge_encode(frame, tok, cfg["max_length"])
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"], foreach=False)
    schedule = get_linear_schedule_with_warmup(opt, math.ceil(total * cfg["warmup_ratio"]), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16 and device.type == "cuda"), init_scale=2048.)
    if saved is not None:
        opt.load_state_dict(saved.pop("optimizer"))
        schedule.load_state_dict(saved.pop("scheduler"))
        scaler.load_state_dict(saved.pop("scaler"))
        _bge_restore_rng(saved.pop("rng"), device)
        del saved
    last_commit = cursor
    started = time.perf_counter()
    weight_id = None

    def commit(complete):
        nonlocal weight_id
        payload = {"signature": signature, "weights": _bge_weights(model), "complete": complete,
                   "cursor": cursor, "applied": applied, "total": total,
                   "train_seconds": train_seconds + time.perf_counter() - started}
        if not complete:
            payload.update(optimizer=opt.state_dict(), scheduler=schedule.state_dict(), scaler=scaler.state_dict(),
                           rng=_bge_rng(device))
        else:
            weight_id = _bge_weight_id(payload["weights"])
            payload["weight_id"] = weight_id
        _bge_atomic(path, lambda stream: torch.save(payload, stream))
        if complete:
            _bge_report(path.parent / "trained.json", {"signature": signature, "weight_id": weight_id})
        progress("train_checkpoint", committed_steps=cursor, total_steps=total)

    for epoch in range(cursor // per_epoch, cfg["epochs"]):
        rng = np.random.default_rng(np.random.SeedSequence([cfg["seed"], fold, epoch]))
        order = rng.permutation(len(groups))
        first_block = cursor % per_epoch
        for block_number in range(first_block, per_epoch):
            if stop_requested():
                if cursor > last_commit:
                    commit(False)
                return False, {"train_steps": applied, "train_seconds": train_seconds + time.perf_counter() - started}
            block = order[block_number * group_batch:(block_number + 1) * group_batch]
            # FP16 overflow retries the same queries and dropout draws at the
            # reduced GradScaler scale. A skipped update never advances the cursor.
            before = _bge_rng(device) if scaler.is_enabled() else None
            for attempt in range(8):
                opt.zero_grad(set_to_none=True)
                if attempt:
                    _bge_restore_rng(before, device)
                for group_index in block:
                    loss = _bge_query_loss(model, tokens, groups[group_index], labels, cfg, tok.pad_token_id, device, dtype)
                    if not torch.isfinite(loss.detach()).item():
                        raise FloatingPointError("Non-finite query loss; previous checkpoint retained.")
                    scaler.scale(loss / len(block)).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg["max_grad_norm"], error_if_nonfinite=not scaler.is_enabled())
                scale = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() >= scale:
                    break
            else:
                raise FloatingPointError("Repeated FP16 overflow at the same optimizer block.")
            schedule.step()
            applied += 1
            cursor += 1
            opt.zero_grad(set_to_none=True)
            if cursor == total or cursor % cfg["checkpoint_steps"] == 0 or cursor % per_epoch == 0:
                commit(cursor == total)
                last_commit = cursor
                print(f"BGE fold {fold}: {cursor}/{total} optimizer steps committed", flush=True)
    return True, {"train_steps": applied, "train_seconds": train_seconds + time.perf_counter() - started, "weight_id": weight_id}


def _bge_read_scores(path, signature, ids):
    with np.load(path, allow_pickle=False) as data:
        if data["signature"].item() != signature or not np.array_equal(data["ids"], ids):
            raise ValueError(f"Prediction signature or ID order mismatch in {path}.")
        scores = data["scores"]
        if scores.shape != ids.shape or not np.isfinite(scores).all():
            raise ValueError(f"Incomplete or non-finite predictions in {path}.")
        return scores


def _bge_save_scores(path, signature, ids, scores):
    if scores.shape != ids.shape or not np.isfinite(scores).all():
        raise ValueError("Refusing to save incomplete/non-finite predictions.")
    _bge_atomic(path, lambda stream: np.savez(stream, signature=signature, ids=ids, scores=scores))


@torch.inference_mode()
def _bge_predict(model, frame, tok, cfg, device, dtype, signature, root, split, stop_requested, progress):
    ids = frame.id.to_numpy(dtype=str)
    final = root / f"{split}.npz"
    if final.is_file():
        return _bge_read_scores(final, signature, ids)
    if stop_requested():
        return None
    model.eval()
    tokens = _bge_encode(frame, tok, cfg["max_length"])
    order = np.asarray(sorted(range(len(ids)), key=lambda i: (len(tokens[i]), ids[i])), dtype=np.int64)
    scores = np.empty(len(ids), dtype=np.float32)
    directory = root / f"{split}_chunks"
    directory.mkdir(exist_ok=True)
    for start in range(0, len(order), cfg["prediction_chunk"]):
        positions = order[start:start + cfg["prediction_chunk"]]
        path = directory / f"{start:07d}.npz"
        if path.is_file():
            scores[positions] = _bge_read_scores(path, signature, ids[positions])
            continue
        for off in range(0, len(positions), cfg["predict_batch"]):
            if stop_requested():
                return None
            indices = positions[off:off + cfg["predict_batch"]]
            values, mask = _bge_pad([tokens[i] for i in indices], tok.pad_token_id, device)
            with torch.autocast(device.type, dtype=dtype, enabled=device.type == "cuda"):
                pred = model(input_ids=values, attention_mask=mask).logits.float().flatten()
            scores[indices] = pred.cpu().numpy()
        _bge_save_scores(path, signature, ids[positions], scores[positions])
        progress("prediction_chunk", split=split, committed_rows=start + len(positions), total_rows=len(ids))
    _bge_save_scores(final, signature, ids, scores)
    return _bge_read_scores(final, signature, ids)


def run_bge_fold(*, fold, train_frame, valid_frame, test_frame, model_dir, artifact_dir,
                 data_signature, code_signature, stop_requested, config=None, on_progress=None):
    """Train and score one fold. Prepared frames retain the original row order.

    Returns arrays aligned to valid_frame/test_frame and a cooperative paused
    flag as status. The caller chooses when to start the next fold. The caller
    supplies hashes of all notebook code and the input CSV manifest, and a
    stop_requested() function that reserves time for one atomic checkpoint.
    """
    cfg = dict(BGE_NIGHT_CONFIG)
    if config:
        if set(config) - set(cfg):
            raise ValueError(f"Unknown BGE parameters: {set(config) - set(cfg)}")
        cfg.update(config)
    if cfg["model"] != BGE_NIGHT_CONFIG["model"] or cfg["revision"] != BGE_NIGHT_CONFIG["revision"]:
        raise ValueError("Unsupported model revision for this BGE component.")
    if (not 0 <= fold < 5 or not data_signature or not code_signature or
            cfg["precision"] != "bf16_if_supported" or not 1 <= cfg["top_layers"] < 24):
        raise ValueError("Invalid BGE fold, precision, top-layer count or missing signatures.")
    for name in ("epochs", "query_batch", "sequence_microbatch", "predict_batch", "prediction_chunk", "checkpoint_steps"):
        if type(cfg[name]) != int or cfg[name] < 1:
            raise ValueError(f"{name} must be a positive integer.")
    if not 8 <= cfg["max_length"] <= 8192 or not 0 <= cfg["warmup_ratio"] < 1:
        raise ValueError("Invalid BGE length or warmup.")
    out = {"fold": int(fold), "status": "paused", "valid_scores": None, "test_scores": None}
    if stop_requested():
        return out
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required for the nightly BGE component.")
    device = torch.device("cuda", torch.cuda.current_device())
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_dir = Path(model_dir)
    for name, ref in _BGE_FILES.items():
        path = (model_dir / name).resolve()
        stat = path.stat()
        _bge_verified_file(str(path), stat.st_size, stat.st_mtime_ns, ref)
    if set(train_frame.query_id) & set(valid_frame.query_id):
        raise ValueError("A validation query appears in BGE training.")
    version_names = ("torch", "transformers", "numpy", "pandas", "safetensors", "tokenizers")
    manifest = {"component": "bge-grouped-v1", "fold": int(fold), "config": cfg,
                "data_signature": str(data_signature), "code_signature": str(code_signature),
                "versions": {name: metadata.version(name) for name in version_names},
                "gpu": torch.cuda.get_device_name(device), "amp_dtype": str(dtype), "cuda": str(torch.version.cuda),
                "base_files": _BGE_FILES, "train": _bge_frame_hash(train_frame, True),
                "valid": _bge_frame_hash(valid_frame, False), "test": _bge_frame_hash(test_frame, False)}
    signature = hashlib.sha256(json.dumps(manifest, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    root = Path(artifact_dir) / f"bge_fold{fold}_{signature[:20]}"
    root.mkdir(parents=True, exist_ok=True)
    out.update(run_dir=str(root), signature=signature)
    def progress(phase, **items):
        event = {"fold": int(fold), "phase": phase, "run_dir": str(root), **items}
        _bge_report(root / "progress.json", event)
        if on_progress is not None:
            on_progress(event)

    with FileLock(str(root / ".lock"), timeout=0):
        _bge_report(root / "manifest.json", manifest)
        trained = root / "trained.json"
        if trained.is_file():
            marker = json.loads(trained.read_text(encoding="utf-8"))
            if marker["signature"] != signature or not (root / "training.pt").is_file():
                raise ValueError("Trained model marker does not match the training checkpoint.")
            prediction_signature = signature + ":" + marker["weight_id"]
            for split, frame in (("valid", valid_frame), ("test", test_frame)):
                path = root / f"{split}.npz"
                if path.is_file():
                    out[f"{split}_scores"] = _bge_read_scores(path, prediction_signature, frame.id.to_numpy(dtype=str))
        if out["valid_scores"] is not None and out["test_scores"] is not None:
            out["status"] = "complete"
            if (root / "stats.json").is_file():
                out["stats"] = json.loads((root / "stats.json").read_text(encoding="utf-8"))
            return out
        if stop_requested():
            return out
        seed = (cfg["seed"] + fold) % 2**32
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.reset_peak_memory_stats(device)
        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False, use_fast=True)
        model = _bge_model(model_dir, cfg, device, dtype)
        try:
            done, stats = _bge_train(model, train_frame, tokenizer, cfg, fold, device, dtype, signature,
                                     root / "training.pt", stop_requested, progress)
            if done:
                prediction_signature = signature + ":" + stats["weight_id"]
                for split, frame in (("valid", valid_frame), ("test", test_frame)):
                    if out[f"{split}_scores"] is None:
                        out[f"{split}_scores"] = _bge_predict(model, frame, tokenizer, cfg, device, dtype,
                                                              prediction_signature, root, split, stop_requested, progress)
                    if out[f"{split}_scores"] is None:
                        break
                if out["valid_scores"] is not None and out["test_scores"] is not None:
                    out["status"] = "complete"
            previous_stats = root / "stats.json"
            previous = json.loads(previous_stats.read_text(encoding="utf-8")) if previous_stats.is_file() else {}
            stats.update(peak_allocated_bytes=max(previous.get("peak_allocated_bytes", 0), torch.cuda.max_memory_allocated(device)),
                         peak_reserved_bytes=max(previous.get("peak_reserved_bytes", 0), torch.cuda.max_memory_reserved(device)),
                         status=out["status"])
            _bge_report(root / "stats.json", stats)
            out["stats"] = stats
            return out
        finally:
            del model
            torch.cuda.empty_cache()

## Семантическая близость Giga-Embeddings

In [4]:
"""Frozen Giga 3B cosine similarity. Call prepare explicitly for an online download."""

import gc
import hashlib
import json
import sqlite3
from contextlib import closing
from pathlib import Path

import numpy as np
import torch
from filelock import FileLock
from huggingface_hub import snapshot_download
from transformers import PreTrainedTokenizerFast, Qwen3Config, Qwen3Model


GIGA_REPOSITORY = "ai-sage/Giga-Embeddings-instruct-3B-0826"
GIGA_REVISION = "b71168088212f0a13688514e4fc288a86106c4e4"
GIGA_FILES = {
    "config.json": "c97518709921dda9a03309fc90cd53b1fa2d837c8e0afd73e8b32a80dd2ebb82",
    "model.safetensors": "de8519bef7ee360043970b0081088c5294e2a9196ad2d0570a33c4f51bb2e134",
    "tokenizer.json": "6fb1280bd7fd529f425929b5df823a5a44485cd7fa9679d1ec1acaac4962e8ca",
    "tokenizer_config.json": "843eeba481465c1485a5b5f24bd24d6c12c4e502c16f093c3ab6a0f058c2c5f2",
}
GIGA_QUERY_PREFIX = "Instruct: Given a query, retrieve relevant passages\nQuery: "
GIGA_MAX_LENGTH = 512
GIGA_DIM = 2048


def _giga_verify(model_dir):
    path = Path(model_dir).resolve()
    for name, wanted in GIGA_FILES.items():
        with (path / name).open("rb") as source:
            actual = hashlib.file_digest(source, "sha256").hexdigest()
        if actual != wanted:
            raise ValueError(f"Giga model file differs from pinned revision: {name}")
    return path


def night_giga_prepare(model_dir):
    """Return a verified local snapshot. This is the only function that can use the network."""
    path = Path(model_dir).resolve()
    if not all((path / name).is_file() for name in GIGA_FILES):
        snapshot_download(repo_id=GIGA_REPOSITORY, revision=GIGA_REVISION, local_dir=path,
                          allow_patterns=list(GIGA_FILES), max_workers=2)
    return _giga_verify(path)


def _giga_load(path, dtype):
    raw = json.loads((path / "config.json").read_text(encoding="utf-8"))
    if raw["model_type"] != "qwen3_bidirec" or raw["is_causal"] or set(raw["layer_types"]) != {"full_attention"}:
        raise ValueError("Unexpected Giga architecture")
    tokenizer = PreTrainedTokenizerFast.from_pretrained(path, local_files_only=True, padding_side="right")
    cfg = Qwen3Config.from_dict(raw)
    model, info = Qwen3Model.from_pretrained(path, config=cfg, dtype=dtype, attn_implementation="sdpa",
                                              local_files_only=True, use_safetensors=True, output_loading_info=True)
    if any(info.get(k) for k in ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs")):
        raise ValueError(f"Giga checkpoint did not load completely: {info}")
    for layer in model.layers:
        layer.self_attn.is_causal = False
    return tokenizer, model.eval().to("cuda")


def _giga_embed(model, ids, token_mask):
    dtype = model.get_input_embeddings().weight.dtype
    key_mask = (1.0 - token_mask[:, None, None, :].to(dtype)) * torch.finfo(dtype).min
    mask = key_mask.expand(-1, 1, ids.shape[1], -1)
    hidden = model(input_ids=ids, attention_mask={"full_attention": mask}, use_cache=False).last_hidden_state
    weights = token_mask.unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1e-6)
    return torch.nn.functional.normalize(pooled, dim=-1).float()


def _giga_inputs(frame):
    names = ["query_clean", "title_clean", "text_clean"]
    if frame[names].isna().any().any():
        raise ValueError("Prepare missing titles as empty strings before Giga scoring.")
    q = (GIGA_QUERY_PREFIX + frame[names[0]]).tolist()
    d = (frame[names[1]] + " | " + frame[names[2]]).tolist()
    texts = list(dict.fromkeys(q + d))
    positions = {value: i for i, value in enumerate(texts)}
    return texts, np.fromiter((positions[x] for x in q), dtype=np.int64), np.fromiter((positions[x] for x in d), dtype=np.int64)


def _giga_cached(db, keys, vectors):
    missing = []
    for start in range(0, len(keys), 400):
        part = keys[start:start + 400]
        slots = {key: start + i for i, key in enumerate(part)}
        marks = ",".join("?" for _ in part)
        for key, blob, digest in db.execute(f"SELECT key, vector, digest FROM vectors WHERE key IN ({marks})", part):
            if len(blob) == 4 * GIGA_DIM and hashlib.sha256(blob).digest() == digest:
                vectors[slots.pop(key)] = np.frombuffer(blob, dtype="<f4")
        missing.extend(slots.values())
    return missing


def _giga_compute(texts, missing, keys, vectors, db, path, dtype, stop_requested):
    tokenizer, model = _giga_load(path, dtype)
    max_batch = 16
    max_tokens = 2048
    completed = len(texts) - len(missing)
    try:
        with torch.inference_mode():
            for offset in range(0, len(missing), 384):
                if stop_requested():
                    return False
                indices = missing[offset:offset + 384]
                encoded = tokenizer([texts[i] for i in indices], truncation=True, max_length=GIGA_MAX_LENGTH,
                                    padding=False, return_token_type_ids=False)["input_ids"]
                order = sorted(range(len(indices)), key=lambda j: len(encoded[j]))
                cursor = 0
                while cursor < len(order):
                    if stop_requested():
                        return False
                    group = order[cursor:cursor + max_batch]
                    while len(group) > 1 and len(group) * len(encoded[group[-1]]) > max_tokens:
                        group = group[:-1]
                    items = [{"input_ids": encoded[j]} for j in group]
                    batch = tokenizer.pad(items, padding=True, return_attention_mask=True, return_tensors="pt").to("cuda")
                    try:
                        values = _giga_embed(model, batch.input_ids, batch.attention_mask).cpu().numpy()
                    except torch.cuda.OutOfMemoryError:
                        if len(group) == 1:
                            raise
                        max_batch = max(1, len(group) // 2)
                        values = None
                    finally:
                        del batch
                    if values is None:
                        gc.collect()
                        torch.cuda.empty_cache()
                        continue
                    if values.shape != (len(group), GIGA_DIM) or not np.isfinite(values).all():
                        raise ValueError("Invalid Giga embeddings")
                    records = []
                    for j, vector in zip(group, values):
                        idx = indices[j]
                        vectors[idx] = vector
                        blob = vector.astype("<f4", copy=False).tobytes()
                        records.append((keys[idx], blob, hashlib.sha256(blob).digest()))
                    with db:
                        db.executemany("INSERT OR REPLACE INTO vectors(key, vector, digest) VALUES (?, ?, ?)", records)
                    completed += len(group)
                    cursor += len(group)
                print(f"Giga embeddings: {completed:,}/{len(texts):,} unique texts", flush=True)
        return True
    finally:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()


def night_giga_scores(frame, model_dir, cache_dir, code_signature, stop_requested):
    """Return one float32 cosine per input row, or None at a requested durable pause."""
    from importlib.metadata import version
    if stop_requested():
        return None
    if frame.empty:
        return np.empty(0, dtype=np.float32)
    if not torch.cuda.is_available():
        raise RuntimeError("Giga scoring requires a CUDA GPU.")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    recipe = {"model": GIGA_REPOSITORY, "revision": GIGA_REVISION, "files": GIGA_FILES, "code": str(code_signature),
              "format": [GIGA_QUERY_PREFIX, " | "], "pool": "masked-mean-l2", "length": GIGA_MAX_LENGTH,
              "dtype": str(dtype), "gpu": torch.cuda.get_device_name(), "torch": str(torch.__version__),
              "transformers": version("transformers"), "tokenizers": version("tokenizers"), "schema": 1}
    namespace = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()
    directory = Path(cache_dir) / "giga3b" / namespace
    directory.mkdir(parents=True, exist_ok=True)
    texts, q_idx, d_idx = _giga_inputs(frame)
    keys = [hashlib.sha256(text.encode("utf-8")).digest() for text in texts]
    vectors = np.empty((len(texts), GIGA_DIM), dtype=np.float32)
    with FileLock(directory / "writer.lock", timeout=0), closing(sqlite3.connect(directory / "embeddings.sqlite")) as db:
        db.execute("PRAGMA synchronous=FULL")
        db.execute("CREATE TABLE IF NOT EXISTS vectors(key BLOB PRIMARY KEY, vector BLOB NOT NULL, digest BLOB NOT NULL) WITHOUT ROWID")
        missing = _giga_cached(db, keys, vectors)
        print(f"Giga: {len(texts)-len(missing):,} cached / {len(texts):,} unique texts.", flush=True)
        if stop_requested():
            return None
        if missing:
            path = _giga_verify(model_dir)
            if stop_requested() or not _giga_compute(texts, missing, keys, vectors, db, path, dtype, stop_requested):
                return None
    scores = np.empty(len(frame), dtype=np.float32)
    for start in range(0, len(frame), 512):
        if stop_requested():
            return None
        stop = min(start + 512, len(frame))
        scores[start:stop] = np.einsum("ij,ij->i", vectors[q_idx[start:stop]], vectors[d_idx[start:stop]])
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite Giga similarity")
    return scores

## Две инструкции для Qwen3-Reranker-4B

In [5]:
RERANK_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements '
                'based on the Query and the Instruct provided. Note that the answer '
                'can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')

RERANK_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

def compact_yes_no_head(base, token_ids):
    """Replace the vocabulary head by its exact frozen no/yes rows.

    The decoder and input vocabulary are unchanged. Only score_batch is used for
    forward inference; generating language-model text with this model is unsupported.
    """
    import torch
    original = base.get_output_embeddings()
    if original.weight.ndim != 2 or not original.weight.is_floating_point():
        raise RuntimeError("The original lm_head must be a floating-point matrix.")
    indices = torch.as_tensor(token_ids, dtype=torch.long, device=original.weight.device)
    weight = original.weight.index_select(0, indices).detach().clone()
    bias = None if original.bias is None else original.bias.index_select(0, indices).detach().clone()
    compact = torch.nn.Linear(weight.shape[1], 2, bias=bias is not None, device=weight.device, dtype=weight.dtype)
    compact.weight = torch.nn.Parameter(weight, requires_grad=False)
    if bias is not None:
        compact.bias = torch.nn.Parameter(bias, requires_grad=False)
    base.set_output_embeddings(compact)
    base._ranking_compact_head = True

def score_batch(model, tokenized):
    """Differentiable logit(yes)-logit(no), without the full vocabulary tensor.

    Explicit projection LoRA layers are installed inside base.model. Calling that
    decoder retains their gradients; only the expensive full lm_head is bypassed.
    """
    import torch
    import torch.nn.functional as functional
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    hidden = base.model(**tokenized, use_cache=False, return_dict=True).last_hidden_state
    mask = tokenized["attention_mask"]
    positions = (mask.long() * torch.arange(mask.shape[1], device=mask.device)).max(dim=1).values
    last = hidden[torch.arange(len(hidden), device=hidden.device), positions]
    head = base.get_output_embeddings()
    if head.weight.ndim != 2 or not head.weight.is_floating_point():
        raise RuntimeError("Expected an unquantized, floating-point language-model output head.")
    indices = torch.tensor(getattr(model, "_ranking_head_rows", model._ranking_token_ids),
                           dtype=torch.long, device=head.weight.device)
    weight = head.weight.index_select(0, indices)
    bias = None if head.bias is None else head.bias.index_select(0, indices)
    with torch.autocast(device_type=last.device.type, enabled=False):
        logits = functional.linear(last.float(), weight.float(), None if bias is None else bias.float())
    return logits[:, 1] - logits[:, 0]

import gc
from pathlib import Path

QWEN_NIGHT_REVISION = "22e683669bc0f0bd69640a1354a6d0aebcfeede5"
QWEN_TASK = ("Given a question in Russian, judge whether the encyclopedic passage contains the direct answer "
             "to the question (the fact itself, not only the same topic or the same name)")


def night_qwen_scores(frame, model_dir, cache_dir, code_signature, stop_requested, *, broad=False):
    """Score a fixed, ordered table. Save length-sorted progress and restore by row identity."""
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    task = "Given a web search query, retrieve relevant passages that answer the query" if broad else QWEN_TASK
    signature = hashlib.sha256((code_signature + QWEN_NIGHT_REVISION + task +
                                _bge_frame_hash(frame, False)).encode()).hexdigest()
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / (signature + ".npz")
    scores = np.full(len(frame), np.nan, dtype="float32")
    if path.exists():
        with np.load(path, allow_pickle=False) as cache:
            stored = cache["scores"]
            if cache["signature"].item() != signature or stored.shape != scores.shape or np.isinf(stored).any():
                raise ValueError("Unexpected Qwen cache contents")
            scores[:] = stored
    todo = np.flatnonzero(np.isnan(scores))
    if not len(todo):
        return scores
    if stop_requested():
        return None
    tokenizer = AutoTokenizer.from_pretrained(model_dir, padding_side="right", local_files_only=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    prefix, suffix = [tokenizer.encode(s, add_special_tokens=False) for s in (RERANK_PREFIX, RERANK_SUFFIX)]
    max_body = (512 if broad else 768) - len(prefix) - len(suffix)
    documents = frame.iloc[todo]
    seq = {}
    for offset in range(0, len(documents), 256):
        if stop_requested():
            return None
        block = documents.iloc[offset:offset + 256]
        inputs = []
        for q, t, x in block[["query_clean", "title_clean", "text_clean"]].itertuples(index=False, name=None):
            title = f"{t}. " if broad else f"Title: {t}\n"
            doc = (title if t else "") + x
            inputs.append(f"<Instruct>: {task}\n<Query>: {q}\n<Document>: {doc}")
        bodies = tokenizer(inputs, add_special_tokens=False, max_length=max_body, truncation=True,
                           return_attention_mask=False)["input_ids"]
        for pos, tokens in zip(todo[offset:offset + len(block)], bodies):
            seq[pos] = {"input_ids": prefix + tokens + suffix}
    order = sorted(seq, key=lambda i: len(seq[i]["input_ids"]))
    device = torch.device("cuda", 0)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(model_dir, local_files_only=True, torch_dtype=dtype,
                    quantization_config=quant, device_map={"": 0}, attn_implementation="sdpa").eval()
    try:
        labels = [tokenizer.encode(w, add_special_tokens=False) for w in ("no", "yes")]
        if any(len(v) != 1 for v in labels) or labels[0] == labels[1]:
            raise ValueError("Qwen yes/no token mismatch")
        model._ranking_token_ids = (labels[0][0], labels[1][0])
        model._ranking_head_rows = (0, 1)
        compact_yes_no_head(model, model._ranking_token_ids)
        start = written = 0
        batch_cap = 24
        began = time.monotonic()
        with torch.inference_mode():
            while start < len(order):
                if stop_requested():
                    if start > written:
                        _bge_atomic(path, lambda f: np.savez(f, signature=signature, scores=scores))
                    return None
                size = min(batch_cap, max(1, 5600 // len(seq[order[start]]["input_ids"])))
                rows = order[start:start + size]
                oom = False
                try:
                    batch = tokenizer.pad([seq[i] for i in rows], return_tensors="pt", padding=True).to(device)
                    with torch.autocast("cuda", dtype=dtype):
                        values = score_batch(model, batch).float().cpu().numpy()
                    del batch
                except torch.cuda.OutOfMemoryError:
                    if size == 1:
                        raise
                    oom = True
                if oom:
                    batch = None
                    batch_cap = max(1, size // 2)
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                if not np.isfinite(values).all():
                    raise FloatingPointError("Qwen returned a nonfinite value")
                scores[rows] = values
                start += len(rows)
                if start - written >= 256 or start == len(order):
                    _bge_atomic(path, lambda f: np.savez(f, signature=signature, scores=scores))
                    written = start
                    remain = (len(order) - start) * (time.monotonic() - began) / start
                    print(f"Qwen {len(scores) - len(order) + start}/{len(scores)}, ETA {remain/60:.1f} мин", flush=True)
        return scores
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

In [6]:
import os
import time
import json
import zipfile
from pathlib import Path
from importlib import metadata
from filelock import FileLock
from huggingface_hub import snapshot_download

def night_data_path(notebook):
    filename = 'amml-ranking-competition-2026.zip'
    if os.getenv('RANK_DATA'):
        return Path(os.environ['RANK_DATA']).expanduser().resolve(strict=True)
    for path in (notebook.parent / filename, Path.cwd() / filename, Path.home() / 'Downloads' / filename):
        if path.is_file():
            return path
    try:
        from google.colab import files
        print(f'Загрузи исходный ZIP с данными соревнования: {filename}')
        uploaded = files.upload()
        archives = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Нужно загрузить ровно один ZIP-архив с train.csv, test.csv и sample_submission.csv.')
        import io
        target = Path.cwd() / filename
        target.write_bytes(archives[0][1])
        print('Архив данных сохранён:', target)
        return target
    except ImportError:
        raise FileNotFoundError(f'Положи {filename} рядом с ноутбуком или задай RANK_DATA.')

def night_model_path(kind, notebook):
    work = notebook.parent.parent.parent / 'work'
    if os.getenv('RANK_' + kind.upper()):
        return Path(os.environ['RANK_' + kind.upper()]).expanduser().resolve(strict=True)
    if kind == 'giga':
        return night_giga_prepare(work / 'night-giga-3b' if work.is_dir() else notebook.parent.parent / 'night_models' / 'giga')
    if kind == 'bge':
        if (work / 'bge-953dc6f6' / 'model.safetensors').is_file():
            return work / 'bge-953dc6f6'
        repo, revision, patterns = (BGE_NIGHT_CONFIG['model'], BGE_NIGHT_CONFIG['revision'], list(_BGE_FILES))
    else:
        repo, revision = ('Qwen/Qwen3-Reranker-4B', QWEN_NIGHT_REVISION)
        patterns = ['*.json', '*.safetensors', '*.txt', '*.jinja']
    for offline in (True, False):
        try:
            directory = Path(snapshot_download(repo_id=repo, revision=revision, allow_patterns=patterns, local_files_only=offline))
            if kind == 'qwen':
                index = json.loads((directory / 'model.safetensors.index.json').read_text(encoding='utf-8'))
                if not all(((directory / f).is_file() for f in set(index['weight_map'].values()))):
                    raise FileNotFoundError('Incomplete Qwen snapshot')
            return directory
        except (OSError, FileNotFoundError):
            if not offline:
                raise

def night_partition(train):
    folds, groups = build_group_folds(train, 5)
    inner = [np.asarray(valid) for _, valid in folds[:4]]
    outer = np.asarray(folds[4][1])
    components = sorted(set(groups[outer]), key=lambda k: hashlib.sha256(('night-audit-v1:' + k).encode()).digest())
    audit_components = set(components[::3])
    audit = np.asarray([g in audit_components for g in groups[outer]])
    if not audit.any() or audit.all():
        raise ValueError('Outer split needs at least two disconnected components')
    return (inner, outer, audit)

In [7]:
import importlib.util, sys, io, gc, time, platform, warnings
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
EXTRA_MANIFEST={'ru': {'repo': 'n-bor1sov/RuModernBERT-base-reranker-ru', 'revision': '46c58c8605609316d3cb9fae2aa02c72c635adae', 'files': [{'rfilename': 'README.md', 'blobId': '436777726f402aa1e60c57364eef3600e8ba645b', 'size': 7694}, {'rfilename': 'config.json', 'blobId': '7db60b5e50e0d44db9972e0141afe33e9ab99291', 'size': 2361}, {'rfilename': 'model.safetensors', 'blobId': '46b6cdb82b93e83d9623c75ec4acdb86b4eb52dc', 'size': 598436708, 'lfs': {'sha256': '0b0b06180c6e5373e93914058fc65c71b77a78621ca6382005fc88dcb59f40ad', 'size': 598436708, 'pointerSize': 134}}, {'rfilename': 'special_tokens_map.json', 'blobId': '747046c366f1b5572ebd114b85812145c47da1e6', 'size': 837}, {'rfilename': 'tokenizer.json', 'blobId': 'f0359060518de3c48acffd08caf39b67e6bd7fcc', 'size': 4753524}, {'rfilename': 'tokenizer_config.json', 'blobId': '89df40bd27ad5fb8886c1e0b720b8dc8e1674413', 'size': 20963}]}, 'jina': {'repo': 'jinaai/jina-reranker-v3.5', 'revision': 'e8a93f33f0b22108f8c2364f8484ce3422552fbc', 'files': [{'rfilename': 'README.md', 'blobId': 'f3a662012e41d0e4399c50a72c522dca0a08f666', 'size': 6678}, {'rfilename': 'added_tokens.json', 'blobId': '88caf7b38f31117f8a9531f32f48cd8535564c37', 'size': 795}, {'rfilename': 'config.json', 'blobId': 'de22a7a575b649ae96abf98a6ce719936b744737', 'size': 1516}, {'rfilename': 'generation_config.json', 'blobId': '66931f13d4291da1e362ac2ac2365c157c327cef', 'size': 202}, {'rfilename': 'merges.txt', 'blobId': '31349551d90c7606f325fe0f11bbb8bd5fa0d7c7', 'size': 1671853}, {'rfilename': 'model.safetensors', 'blobId': 'd71ef8efdb718c5208590f6d6511e430f9763034', 'size': 1193708152, 'lfs': {'sha256': '50c684b703a1770389cd2992c6ca3c5f76ad0a270e975374c182cde3e5dfb6fe', 'size': 1193708152, 'pointerSize': 135}}, {'rfilename': 'modeling.py', 'blobId': '4d1d4668e6da0b159548d6cb0a3edaff407fc730', 'size': 10389}, {'rfilename': 'special_tokens_map.json', 'blobId': '5db181d71f801f6648ddc9acc2e24d3560365419', 'size': 777}, {'rfilename': 'tokenizer.json', 'blobId': '9b3649cfc84ba610ee70cb5f6fb68600fd4253d8', 'size': 11423225, 'lfs': {'sha256': '4e95945ab0cef486709f760b81efcc7a6e75747f9165d13ead29159737455803', 'size': 11423225, 'pointerSize': 133}}, {'rfilename': 'tokenizer_config.json', 'blobId': '3ef92c4049184e21017c3e8c7de88921bfbba453', 'size': 10723}, {'rfilename': 'vocab.json', 'blobId': '4783fe10ac3adce15ac8f358ef5462739852c569', 'size': 2776833}]}}

In [8]:
def verified(path, metadata):
    if not path.is_file() or path.stat().st_size != metadata["size"]:
        return False
    if metadata.get("lfs"):
        with path.open("rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == metadata["lfs"]["sha256"]
    digest = hashlib.sha1(f"blob {metadata['size']}\0".encode())
    digest.update(path.read_bytes())
    return digest.hexdigest() == metadata["blobId"]

In [9]:
def sanitized_batch(tok, texts, limit):
    encoded = tok(texts, add_special_tokens=False, truncation=True, max_length=limit)["input_ids"]
    return [text if len(tokens) < limit else tok.decode(tokens, skip_special_tokens=True)
            for text, tokens in zip(texts, encoded)]

def score_ru(frame, scores, save, folder, tok, device, smoke):
    config = AutoConfig.from_pretrained(folder, local_files_only=True, trust_remote_code=False)
    config.reference_compile = False
    dtype = torch.bfloat16 if device == "cuda" else torch.float32
    model, info = AutoModelForSequenceClassification.from_pretrained(folder, config=config, dtype=dtype,
                            local_files_only=True, attn_implementation="sdpa", output_loading_info=True)
    if info["missing_keys"] or info["mismatched_keys"] or config.num_labels != 1:
        raise RuntimeError(f"Incomplete classification weights: {info}")
    model = model.to(device).eval()
    docs = (frame.title_clean.where(frame.title_clean.eq(""), frame.title_clean + ". ") + frame.text_clean).tolist()
    q = frame.query_clean.tolist()
    pending = np.flatnonzero(~np.isfinite(scores))
    order = sorted(pending, key=lambda i: len(q[i]) + len(docs[i]))
    if smoke:
        order = order[:2]
    for start in range(0, len(order), 16):
        rows = order[start:start + 16]
        batch = tok([q[r] for r in rows], [docs[r] for r in rows], truncation="only_second", max_length=512,
                    padding=True, return_tensors="pt").to(device)
        out = model(**batch).logits.flatten().float().cpu().numpy()
        if out.shape != (len(rows),) or not np.isfinite(out).all():
            raise ValueError("Invalid RuModernBERT scores")
        scores[rows] = out
        if not smoke and (start + len(rows)) // 16 % 32 == 0:
            save()
            print(f"RuModernBERT {np.isfinite(scores).sum()}/{len(scores)} pairs", flush=True)
    if not smoke:
        save()

## Признаки текста и предложения

In [10]:
def historical_features():
    """Features adapted from 0.959.ipynb, cells 3 and 19. Corpus statistics are train-fold fitted."""
    from collections import Counter
    from functools import lru_cache
    import html
    import math
    import re
    import unicodedata
    import numpy as np
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    USE_RRF = True

    TOK = re.compile(r"\w+", re.U)

    _MOJI = re.compile(r"[РС][\u0080-\u00bf\u0400-\u04ff]")

    def fix_moji(s):
        if not isinstance(s, str) or len(_MOJI.findall(s)) <= 5 or re.search(r"[а-яё]{4}", s): return s
        for errors in ("strict", "ignore"):
            try:
                r = s.encode("cp1251", errors=errors).decode("utf-8", errors=errors)
                if re.search(r"[а-яё]{4}", r): return r
            except Exception: pass
        return s

    def norm(s):
        if not isinstance(s, str): return ""
        s = fix_moji(s)
        s = re.sub(r"<[^>]{1,40}>", " ", html.unescape(s))
        s = unicodedata.normalize("NFC", s).replace("́", "").replace("̀", "")
        s = s.lower().replace("ё", "е").replace("\xa0", " ")
        return s

    def norm_light(s):
        if not isinstance(s, str): return ""
        s = fix_moji(s)
        s = re.sub(r"<[^>]{1,40}>", " ", html.unescape(s))
        s = re.sub("[\u00ad\u200b-\u200f\u2060\ufeff\u0301]", "", s).replace("\xa0", " ")
        return s

    @lru_cache(maxsize=24000)
    def stems(s, k=5):
        return [t[:k] for t in TOK.findall(norm(s))]

    def clean_frame(df):
        df = df.copy()
        df["title"] = df["title"].fillna("").map(fix_moji)
        df["text"] = df["text"].map(fix_moji)
        cc = pd.to_numeric(df["article_char_count"].astype(str).str.replace(r"[\s\u202f\xa0]", "", regex=True)
                           .str.replace(",", "."), errors="coerce")
        df["article_char_count"] = cc.where(cc > 0)
        for col in ("article_passage_count", "passage_index"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["article_passage_count"] = df["article_passage_count"].where(df["article_passage_count"] > 0)
        df["passage_index"] = df["passage_index"] - (df["ingest_source"] == "S3").astype(int)
        a = df.groupby("article_id")
        df["article_char_count"] = a["article_char_count"].transform("min")
        df["article_passage_count"] = a["article_passage_count"].transform("max")
        for c in ["passage_index", "article_passage_count", "article_char_count"]:
            df[c + "_na"] = df[c].isna().astype(int)
        return df

    def rowwise_cos(A, B):
        return np.asarray(A.multiply(B).sum(1)).ravel()

    def cap_overlap(q, t):
        qc = set(re.findall(r"\b[А-ЯA-Z][а-яa-z]+", q))
        tc = set(re.findall(r"\b[А-ЯA-Z][а-яa-z]+", t))
        return len(qc & tc) / (len(qc) + 1e-9) if qc else 0

    def build_features(df, context):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        df = clean_frame(df).reset_index(drop=True)
        F = pd.DataFrame(index=df.index)
        qn = df["query"].map(norm); tn = df["text"].map(norm); ttn = df["title"].map(norm)
        for name, vectorizer in context['vectorizers'].items():
            chunks=[]
            for start in range(0,len(df),192):
                positions=slice(start,start+192)
                Q,T,TT=(vectorizer.transform(s.iloc[positions]) for s in (qn,tn,ttn))
                chunks.append(np.column_stack([rowwise_cos(Q,T),rowwise_cos(Q,TT),rowwise_cos(Q,T+TT)]))
            values=np.concatenate(chunks)
            for j,field in enumerate(('text','title','tt')): F[f'cos_{field}_{name}']=values[:,j]

        # BM25 + coverage
        dstems_text = [stems(t) for t in df["text"]]
        dstems_title = [stems(t) for t in df["title"]]
        qstems = [stems(q) for q in df["query"]]
        N,dfc,avg_len=context['N'],context['bm25_df'],context['average_doc_length']
        idf=lambda w: math.log1p((N-dfc.get(w,0)+.5)/(dfc.get(w,0)+.5))
        K1, B = 1.5, 0.5   # для коротких фрагментов b меньше 0.75
        rows = []
        for qs, ds, ts, qraw, tx in zip(qstems, dstems_text, dstems_title, df["query"], tn):
            qset = list(dict.fromkeys(qs)); cnt = Counter(ds); tcnt = Counter(ts)
            L = len(ds) + 1e-9
            w = np.array([idf(x) for x in qset]) if qset else np.zeros(0)
            tfs = np.array([cnt.get(x, 0) for x in qset]); tts = np.array([tcnt.get(x, 0) for x in qset])
            bm = (w * tfs * (K1 + 1) / (tfs + K1 * (1 - B + B * L / avg_len))).sum() if qset else 0
            bmt = (w * tts * (K1 + 1) / (tts + K1)).sum() if qset else 0
            wsum = w.sum() + 1e-9
            cov = (w * (tfs > 0)).sum() / wsum; covt = (w * (tts > 0)).sum() / wsum
            covall = (w * ((tfs + tts) > 0)).sum() / wsum
            n_in = int((tfs > 0).sum()); n_in_t = int((tts > 0).sum())
            head = set(ds[:30]); covh = (w * np.array([x in head for x in qset])).sum() / wsum if qset else 0
            qb = set(zip(qs, qs[1:])); db = set(zip(ds, ds[1:])) | set(zip(ts, ts[1:]))
            bg = len(qb & db) / (len(qb) + 1e-9) if qb else 0
            qdig = set(re.findall(r"\d+", qraw)); ddig = set(re.findall(r"\d+", tx))
            dig = len(qdig & ddig) / len(qdig) if qdig else -1
            rows.append((bm, bmt, cov, covt, covall, n_in, n_in_t, covh, bg, dig,
                         len(qset), len(ds), len(ts), wsum))
        cols = ["bm25","bm25_t","cov","cov_t","cov_all","n_in","n_in_t","cov_head","bigram","digit_ov",
                "q_len","txt_len","title_len","q_idfsum"]
        for c, v in zip(cols, zip(*rows)): F[c] = v
        F["miss_tt"] = F["cov_all"] - F["cov"]
        F["cap_ov"] = [cap_overlap(q, t) for q, t in zip(df["query"], df["text"])]

        # sentence-level
        X2 = {k: [] for k in ["sent_cov_max","sent_bm_max","sent_cov_first","title_cov_by_q","title_exact_in_q","lead_cov",
                              "n_years","n_digits","n_caps","is_def","title_in_first","n_sent","q_in_text_exact"]}
        for qs, qraw, raw, tit in zip(qstems, df["query"], df["text"], df["title"]):
            qset = list(dict.fromkeys(qs)); w = np.array([idf(x) for x in qset]) if qset else np.zeros(1); ws = w.sum() + 1e-9
            t_clean = norm(raw)
            sents = [x for x in re.split(r"(?<=[.!?])\s+", t_clean) if x.strip()] or [t_clean]
            sets = [set(stems(sn)) for sn in sents[:25]]
            covs = [(w * np.array([x in ss for x in qset])).sum()/ws if qset else 0 for ss in sets]
            X2["sent_cov_max"].append(max(covs)); X2["sent_bm_max"].append(np.sort(covs)[-2:].sum())
            X2["sent_cov_first"].append(covs[0]); X2["n_sent"].append(len(sents))
            lead = set(stems(t_clean[:200])); X2["lead_cov"].append((w * np.array([x in lead for x in qset])).sum() / ws if qset else 0)
            ts = [x for x in stems(re.sub(r"\(.*?\)", " ", tit))]
            qsset = set(qs)
            X2["title_cov_by_q"].append(np.mean([x in qsset for x in ts]) if ts else -1)
            tt = re.sub(r"\s+", " ", norm(re.sub(r"\(.*?\)", " ", tit))).strip()
            X2["title_exact_in_q"].append(int(len(tt) > 2 and tt in norm(qraw)))
            X2["n_years"].append(len(re.findall(r"\b(?:1\d{3}|20\d{2})\b", raw))); X2["n_digits"].append(len(re.findall(r"\d+", raw)))
            X2["n_caps"].append(len(re.findall(r"(?<![.!?]\s)\b[А-ЯA-Z][а-яa-z]+", raw[10:])))
            X2["is_def"].append(int(bool(re.search(r"^.{0,150}?(—|-|–|это|являет)", raw))))
            X2["title_in_first"].append(int(len(tt) > 2 and tt[:30] in t_clean[:300]))
            qn_ = re.sub(r"[^\w ]", "", norm(qraw))
            X2["q_in_text_exact"].append(int(len(qn_) > 3 and any(qn_[:-1] in t_clean for _ in [0])))
        F = pd.concat([F, pd.DataFrame(X2)], axis=1)

        qr = df["query"].map(norm)
        qt = pd.DataFrame({
            "qt_kto": qr.str.match(r"^(кто|кого|кем|кому)"), "qt_kogda": qr.str.contains(r"^когда|в каком (году|веке)|какого числа|в какой год|дата"),
            "qt_skolko": qr.str.contains(r"сколько|какой (средний|максимальный|общий)|какая (длина|площадь|протяж|высота|численность)"),
            "qt_gde": qr.str.match(r"^(где|куда|откуда)"), "qt_chto": qr.str.contains(r"^что такое|^кто так|^что значит|^что за"),
            "qt_kak": qr.str.match(r"^(как|каким)"), "qt_pochemu": qr.str.match(r"^(почему|зачем|отчего)"),
            "qt_yesno": qr.str.contains(r"^(является ли|правда ли|был ли|есть ли|можно ли|верно ли)|(ли )"),
            "qt_which": qr.str.match(r"^(какой|какая|какое|какие|каков|чем)"),
        }).astype(int)
        qt["q_words"] = qr.str.split().str.len()
        F = pd.concat([F, qt], axis=1)

        # TF-IDF and truncated SVD are fitted solely on fold-training documents and queries.
        sv,sd=context['lsa_vectorizer'],context['svd']
        def lsa(terms):
            value=sd.transform(sv.transform(terms));return value/(np.linalg.norm(value,axis=1,keepdims=True)+1e-9)
        chunk=[]
        for start in range(0,len(df),256):
            ix=slice(start,start+256); qvec=lsa(qn.iloc[ix]); texts=tn.iloc[ix]; titles=ttn.iloc[ix]
            chunk.append(np.column_stack([(qvec*lsa(part)).sum(1) for part in (texts,texts+' '+titles,titles)]))
        for name,values in zip(('lsa_text','lsa_tt','lsa_title'),np.concatenate(chunk).T):F[name]=values

        # метаданные
        for c in ["passage_index","article_passage_count","article_char_count"]:
            F[c] = df[c]; F[c + "_na"] = df[c + "_na"]
        F["pos_rel"] = df["passage_index"] / df["article_passage_count"]
        F["pos_rel_sq"] = F["pos_rel"] ** 2
        F["pos_rel_inv"] = 1 - F["pos_rel"]
        F["title_na"] = (df["title"] == "").astype(int)
        F["text_chars"] = df["text"].str.len()
        for s in ["S0","S1","S2","S3"]: F["src_" + s] = (df["ingest_source"] == s).astype(int)
        F["title_in_query"] = [int(len(t) > 2 and t in q) for t, q in zip(ttn.str.replace(r"\s*\(.*?\)", "", regex=True).str.strip(), qn)]
        F["has_accent"] = df["text"].fillna("").str.contains("́").astype(int)

        # corpus stats
        F["doc_nq"] = df.groupby("document_id")["query_id"].transform("nunique")
        F["art_nq"] = df.groupby("article_id")["query_id"].transform("nunique")

        # внутри запроса
        g = df["query_id"]
        F["q_n"] = g.map(g.value_counts())
        F["art_cnt_in_q"] = df.groupby(["query_id","article_id"])["document_id"].transform("count")
        F["art_frac_in_q"] = F["art_cnt_in_q"] / F["q_n"]
        F["n_art_in_q"] = df.groupby("query_id")["article_id"].transform("nunique")
        base = [c for c in F.columns if c.startswith(("cos_","bm25","cov","bigram","n_in","miss","sent_","lead_","lsa_","title_cov","cap_ov"))]
        grp = F.groupby(g)
        new = {}
        for c in base:
            new[c + "_rk"] = grp[c].rank(ascending=False, pct=True, method="average")
            new[c + "_dmax"] = F[c] - grp[c].transform("max")
            new[c + "_z"] = (F[c] - grp[c].transform("mean")) / (grp[c].transform("std") + 1e-6)
        F = pd.concat([F, pd.DataFrame(new)], axis=1)

        key = [g, df["article_id"]]
        new = {}
        for c in ["cos_text_w5","cos_title_w5","bm25","cos_tt_c","cov_all","sent_cov_max","lsa_tt","cap_ov"]:
            ag = F.groupby(key)[c]
            new[c + "_artmax"] = ag.transform("max"); new[c + "_artmean"] = ag.transform("mean")
            new[c + "_artrk"] = ag.rank(ascending=False, method="min")
        F = pd.concat([F, pd.DataFrame(new)], axis=1)
        F["log_cc"] = np.log1p(F["article_char_count"])

        # интеракции для коротких запросов
        short = (F["q_len"] <= 2).astype(int)
        F["cov_x_short"] = F["cov"] * short
        F["bm25_x_short"] = F["bm25"] * short
        return F

    _A_TOK = re.compile(r"\w+", re.U)

    _A_YEAR = re.compile(r"\b(?:1\d{3}|20\d{2})\b")

    _A_MONTH = re.compile(r"\b\d{1,2}\s+(?:январ|феврал|март|апрел|ма[яй]|июн|июл|август|сентябр|октябр|ноябр|декабр)\w*", re.I)

    _A_CAP = re.compile(r"\b[А-ЯЁA-Z][а-яёa-z]{2,}")

    _A_SENT = re.compile(r"(?<=[.!?])\s+")

    _A_WHEN = re.compile(r"^(когда|в каком (году|веке|месяце)|в какой год|какого числа|в какие годы)|\bдата\b", re.I)

    _A_HOWMANY = re.compile(r"^(сколько|какова (длина|площадь|высота|численность)|какая (длина|площадь|высота|численность|температура|скорость))|средн\w+ (длина|высота|вес)", re.I)

    _A_WHO = re.compile(r"^(кто|кого|кем|кому|чей|чья|чьё)\b", re.I)

    _A_WHERE = re.compile(r"^(где|куда|откуда|в каком (городе|регионе|стране|государстве))", re.I)

    @lru_cache(maxsize=24000)
    def _a_stems(s, k=5): return [t[:k] for t in _A_TOK.findall(s.lower().replace("ё", "е"))]

    def answer_features(df, context):
        """df: колонки query, title, text, document_id. Индекс не важен, результат выровнен по позиции строк."""
        N,dfc=context['N'],context['answer_df']
        idf = lambda w: math.log(1 + (N - dfc.get(w, 0) + .5) / (dfc.get(w, 0) + .5))
        cols = ["a_best1", "a_best2", "a_best_pos", "a_qtype", "a_date_best", "a_date_pair", "a_num_best", "a_cap_best", "a_ans_match",
                "a_ans_match_x_cov", "a_ent_date_sent", "a_ent_year_dist", "a_ent_title", "a_subj_title_jacc", "a_n_ent"]
        rows = []
        for q, ttl, txt in zip(df["query"].astype(str).values, df["title"].fillna("").astype(str).values, df["text"].fillna("").astype(str).values):
            qs = list(dict.fromkeys(_a_stems(q))); w = np.array([idf(x) for x in qs]) if qs else np.zeros(1); ws = w.sum() + 1e-9
            sents = [s for s in _A_SENT.split(txt) if s.strip()][:40] or [txt]
            sst = [set(_a_stems(s)) for s in sents]
            cov = np.array([(w * np.array([x in ss for x in qs])).sum() / ws if qs else 0.0 for ss in sst])
            b = int(cov.argmax()); best1 = float(cov[b])
            if len(sents) > 1:
                pair = [(w * np.array([x in (sst[i] | sst[i + 1]) for x in qs])).sum() / ws if qs else 0.0 for i in range(len(sents) - 1)]
                pb = int(np.argmax(pair)); best2 = float(pair[pb]); pair_text = sents[pb] + " " + sents[pb + 1]
            else: best2 = best1; pair_text = sents[b]
            bs = sents[b]
            is_date = lambda s: bool(_A_YEAR.search(s) or _A_MONTH.search(s))
            date_best, date_pair = int(is_date(bs)), int(is_date(pair_text))
            num_best = int(bool(re.search(r"\d", bs))); cap_best = int(len(_A_CAP.findall(bs[1:])) > 0)
            if _A_WHEN.search(q): qt = 1; am = date_pair
            elif _A_HOWMANY.search(q): qt = 2; am = int(bool(re.search(r"\d", pair_text)))
            elif _A_WHO.search(q): qt = 3; am = int(len(_A_CAP.findall(pair_text[1:])) > 0)
            elif _A_WHERE.search(q): qt = 4; am = int(len(_A_CAP.findall(pair_text[1:])) > 0)
            else: qt = 0; am = -1
            ents = _A_CAP.findall(q[1:]); es = set(_a_stems(" ".join(ents))); n_ent = len(es)
            ent_date = 0
            if es: ent_date = int(any(len(es & ss) / len(es) >= .5 and is_date(s) for s, ss in zip(sents, sst)))
            words = [x.lower().replace("ё", "е") for x in _A_TOK.findall(txt)]
            pe = [i for i, x in enumerate(words) if x[:5] in es] if es else []; py = [i for i, x in enumerate(words) if _A_YEAR.fullmatch(x)]
            dist = min(abs(i - j) for i in pe for j in py) if pe and py else -1
            ts = set(_a_stems(re.sub(r"\(.*?\)", " ", ttl)))
            subj = es if es else set(qs[:2])
            ent_title = len(es & ts) / len(es) if es and ts else -1
            jac = len(subj & ts) / (len(subj | ts) + 1e-9) if ts and subj else -1
            rows.append((best1, best2, b / max(1, len(sents) - 1), qt, date_best, date_pair, num_best, cap_best, am,
                         (am if am >= 0 else 0) * best2, ent_date, min(dist, 300) if dist >= 0 else -1, ent_title, jac, n_ent))
        A = pd.DataFrame(rows, columns=cols, index=df.index)
        g = df["query_id"].values
        for c in ["a_best1", "a_best2", "a_ans_match_x_cov", "a_subj_title_jacc", "a_ent_title"]:
            s = A[c].groupby(g)
            A[c + "_rk"] = s.rank(ascending=False, pct=True); A[c + "_dmax"] = A[c] - s.transform("max")
        return A

    def ce_block(name, s, qid, aid, docid):
        s = np.asarray(s, float); d = pd.DataFrame({"s": s, "q": qid, "a": aid, "doc": docid})
        gq = d.groupby("q")["s"]; o = {}
        o[name] = s
        o[name + "_rk"] = gq.rank(ascending=False, pct=True).values
        o[name + "_rkabs"] = gq.rank(ascending=False, method="min").values
        o[name + "_dmax"] = s - gq.transform("max").values
        o[name + "_z"] = (s - gq.transform("mean").values) / (gq.transform("std").values + 1e-6)
        e = np.exp(s - gq.transform("max").values)
        o[name + "_sm"] = e / pd.Series(e).groupby(qid).transform("sum").values
        ga = d.groupby(["q","a"])["s"]
        o[name + "_amax"] = ga.transform("max").values; o[name + "_amean"] = ga.transform("mean").values
        o[name + "_arank"] = ga.rank(ascending=False, method="min").values
        o[name + "_doc_mean"] = d.groupby("doc")["s"].transform("mean").values
        return pd.DataFrame(o)

    def build_stage2(X, dfx, ces, zs, extras=None):
        parts = [X.reset_index(drop=True)]
        qid, aid, did = dfx["query_id"].values, dfx["article_id"].values, dfx["document_id"].values
        allr = []; absr = []
        for i, c in enumerate(ces):
            parts.append(ce_block(f"ce{i}", c, qid, aid, did)); allr.append(parts[-1][f"ce{i}_rk"].values); absr.append(parts[-1][f"ce{i}_rkabs"].values)
        if zs is not None:
            parts.append(ce_block("zs", zs, qid, aid, did)); allr.append(parts[-1]["zs_rk"].values); absr.append(parts[-1]["zs_rkabs"].values)
        for en, ev in (extras or {}).items():
            parts.append(ce_block(en, ev, qid, aid, did)); allr.append(parts[-1][en + "_rk"].values); absr.append(parts[-1][en + "_rkabs"].values)
        S2 = pd.concat(parts, axis=1)
        if allr:
            R = np.vstack(allr); S2["ens_rk_mean"] = R.mean(0); S2["ens_rk_min"] = R.min(0); S2["ens_rk_max"] = R.max(0); S2["ens_rk_std"] = R.std(0)
            zsum = sum(S2[c] for c in S2.columns if re.fullmatch(r"(ce\d+|zs|gg|bm3|qw|qj|rr4|rr4b|rr4c|rr8|giga_full|giga_text|giga3_full|giga3_text)_z", c))
            S2["ens_z"] = zsum
            S2["ens_z_rk"] = pd.Series(zsum.values).groupby(qid).rank(ascending=False, pct=True).values
            S2["ens_z_dmax"] = zsum.values - pd.Series(zsum.values).groupby(qid).transform("max").values
        if allr:
            for c in ["bm25_rk","cos_tt_c_rk","cov_all_rk","lsa_tt_rk","cap_ov_rk"]:
                if c in S2.columns: S2[c + "_x_ens"] = S2[c] * S2["ens_rk_mean"]; S2[c + "_m_ens"] = S2[c] - S2["ens_rk_mean"]
        if USE_RRF:
            # RRF (лекция 11): сумма 1/(60+rank) по ранжированиям моделей + лексика
            ranks = list(absr)
            lex = pd.Series(X["bm25"].values).groupby(qid).rank(ascending=False, method="min").values
            for nm, rr in [("rrf_models", ranks), ("rrf_all", ranks + [lex])]:
                if rr:
                    v = sum(1.0 / (60 + r) for r in rr)
                    S2[nm] = v
                    S2[nm + "_rk"] = pd.Series(v).groupby(qid).rank(ascending=False, pct=True).values
                    S2[nm + "_dmax"] = v - pd.Series(v).groupby(qid).transform("max").values
        return S2

    def fit_features(training):
        clean=clean_frame(training).reset_index(drop=True)
        docs=clean.drop_duplicates('document_id'); queries=clean.drop_duplicates('query_id')['query']
        corpus=pd.concat([docs.text,docs.title,queries]).map(norm)
        vec={}
        for name,kwargs in [('w5',dict(analyzer=lambda s: stems(s,5))),('wfull',dict(analyzer=lambda s: stems(s,40))),
                            ('c',dict(analyzer='char_wb',ngram_range=(3,5),min_df=2))]:
            vec[name]=TfidfVectorizer(**kwargs,dtype=np.float32,sublinear_tf=True).fit(corpus)
        lsa_vectorizer=TfidfVectorizer(analyzer=lambda s:stems(s,5),min_df=2,sublinear_tf=True,dtype=np.float32)
        svcorpus=pd.concat([docs.text+' '+docs.title,queries]).map(norm)
        svd=TruncatedSVD(n_components=200,random_state=0).fit(lsa_vectorizer.fit_transform(svcorpus))
        bm25df=Counter();answerdf=Counter()
        for text,title in zip(docs.text,docs.title):bm25df.update(set(stems(text))|set(stems(title)))
        for raw in training.drop_duplicates('document_id').text:answerdf.update(set(_a_stems(norm_light(raw))))
        return dict(N=len(docs),average_doc_length=np.mean([len(stems(t)) for t in clean.text]),bm25_df=bm25df,
                    answer_df=answerdf,vectorizers=vec,lsa_vectorizer=lsa_vectorizer,svd=svd)


    def transform_features(frame,context):
        frame=frame.reset_index(drop=True)
        static=build_features(frame,context)
        for_answer=frame[['query','title','text','document_id','query_id']].copy()
        for_answer['title']=for_answer.title.fillna('').map(fix_moji);for_answer['text']=for_answer.text.map(norm_light)
        answer=answer_features(for_answer,context)
        result=pd.concat([static,answer],axis=1).astype(np.float32)
        if result.columns.duplicated().any() or np.isinf(result.to_numpy()).any():raise ValueError('Invalid historical features')
        return result

    return fit_features,transform_features,build_stage2

historical_fit,historical_transform,historical_stage2=historical_features()

## LambdaRank и XE-NDCG

In [11]:
from lightgbm import LGBMRanker
TUNING={'n_estimators': 350, 'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 30, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.5, 'reg_lambda': 5.0, 'n_jobs': 1, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}

def tree_scores(train,y,x,v):
    order=np.lexsort((train.id.to_numpy(str),train.query_id.to_numpy(str)));q=train.query_id.to_numpy(str)[order]
    group=pd.Series(q).groupby(q,sort=False).size().tolist();result=[]
    for objective,seed,extra in [('lambdarank',322,dict(lambdarank_truncation_level=20)),('rank_xendcg',710,dict(objective_seed=710))]:
        model=LGBMRanker(**TUNING,**extra,random_state=seed,objective=objective).fit(x.iloc[order],y[order],group=group)
        result.append(model.predict(v))
    return result

## Полные проходы Jina и регрессия ожидаемого вклада в NDCG

In [12]:
from threadpoolctl import threadpool_limits
from lightgbm import LGBMRegressor
DATA_SHA='79b7b5291e1f1e6d5c2fc992c0cf5db316230ad5cea7970f4a4d113e44975cd7'
LIGHT_REV='e8a93f33f0b22108f8c2364f8484ce3422552fbc'
LIGHT_BASE_SHA=('fabf06bf7915f4ee7cf9ac5e70a65ad5342c1f0b158638285845dbdfc29e54cb', '82cacef4c830f5487bdc0880dcaf315e998cb4249b19ceba58c5b7f79449c80f', '952c61f3ac3f96964e2e13f0ba5365c85e7713ee1392df5e0fb536c3e67f8b41')
LIGHT_DIRECT_SHA='f50592a431e1c05c9330cd195eb88cc92831a4b27d9bef558936d4dfc7b5f9ef'
LIGHT_REVERSE_SHA='1e9dbc1fc9f21bf38982706a3b181a9085d1f9d3116a320232efb9fba4da8923'

In [13]:
def jina_full_module(folder):
    spec=importlib.util.spec_from_file_location("pinned_jina_full_list",folder/"modeling.py")
    module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
    return module

def jina_full_plan(frame,folder,reverse=False):
    module=jina_full_module(folder);tok=AutoTokenizer.from_pretrained(folder,local_files_only=True)
    special={"query_embed_token":"<|rerank_token|>","doc_embed_token":"<|embed_token|>"}
    ids=frame.id.to_numpy(str)
    docs=(frame.title_clean.where(frame.title_clean.eq(""),frame.title_clean+". ")+frame.text_clean).tolist()
    inputs=[];members=[];same=[];ratio=np.zeros(len(frame),np.float32)
    for g in frame.groupby("query_id",sort=True).indices.values():
        rows=np.asarray(sorted(g,key=lambda j:hashlib.sha256(ids[j].encode()).digest()),dtype=np.int32)
        if reverse:rows=rows[::-1].copy()
        q=frame.query_clean.iat[rows[0]];group=[docs[j] for j in rows]
        packed=tok(group,add_special_tokens=False,truncation=False,verbose=False)["input_ids"]
        sizes=np.array([len(a) for a in packed])
        limit=min(192,1536//len(rows))
        ratio[rows]=np.maximum(0,sizes-limit)/np.maximum(1,sizes)
        raw=module.format_docs_prompts_func(q,group,special_tokens=special)
        seq=tok(raw,add_special_tokens=True,truncation=False,verbose=False)["input_ids"]
        if seq.count(151670)!=len(rows) or seq.count(151671)!=1:raise ValueError("Lost a passage in the full list")
        small=sanitized_batch(tok,group,limit)
        old=module.format_docs_prompts_func(sanitized_batch(tok,[q],96)[0],small,special_tokens=special)
        same.append(seq==tok(old,add_special_tokens=True,truncation=False,verbose=False)["input_ids"])
        inputs.append(np.asarray(seq,dtype=np.int32));members.append(rows)
    if max(map(len,inputs))>131072:raise ValueError("A full query exceeds the pinned model context")
    return inputs,members,np.asarray(same,bool),ratio

def jina_full_load(folder,device):
    module=jina_full_module(folder)
    model,info=module.JinaForRanking.from_pretrained(folder,dtype=torch.bfloat16,attn_implementation="sdpa",
        local_files_only=True,output_loading_info=True)
    if info["mismatched_keys"] or any(k!="lm_head.weight" for k in info["missing_keys"]):raise ValueError("Incomplete listwise model")
    return model.to(device).eval()

@torch.inference_mode()
def jina_full_score(model,seq,n,device,stream=False,return_vectors=False):
    tokens=torch.as_tensor(seq.astype(np.int64),device=device)[None]
    if len(seq)<=8192 and not stream:
        h=model.model(input_ids=tokens,use_cache=False).last_hidden_state
        doc=h[tokens==model.doc_embed_token_id];query=h[tokens==model.query_embed_token_id]
    else:
        # Qwen3 is causal. Keep its KV history for the entire question while evaluating consecutive chunks.
        docs=[];queries=[];cache=None
        for piece in tokens.split(512,dim=1):
            out=model.model(input_ids=piece,past_key_values=cache,use_cache=True)
            cache=out.past_key_values
            docs.append(out.last_hidden_state[piece==model.doc_embed_token_id])
            queries.append(out.last_hidden_state[piece==model.query_embed_token_id])
        doc=torch.cat(docs);query=torch.cat(queries)
    dv=model.projector(doc).float().cpu().numpy().astype(np.float16)
    qv=model.projector(query).float().cpu().numpy().astype(np.float16)
    if dv.shape!=(n,512) or qv.shape!=(1,512):raise ValueError("Full list embedding shape differs")
    d=dv.astype(np.float32);q=qv[0].astype(np.float32)
    score=(d@q)/(np.linalg.norm(d,axis=1)*np.linalg.norm(q))
    if not np.isfinite(score).all():raise ValueError("Nonfinite full-list similarity")
    return (score,dv,qv[0]) if return_vectors else score

In [14]:
def standalone_gain(train,labels,x,held):
    q=train.query_id.reset_index(drop=True)
    npositive=pd.Series(labels).groupby(q,sort=False).transform("sum").to_numpy(dtype=int)
    discounts=np.r_[0.,np.cumsum(1/np.log2(np.arange(2,12)))]
    denom=discounts[np.minimum(npositive,10)]
    target=np.divide(labels,denom,out=np.zeros(len(labels)),where=denom>0)
    weight=1/q.groupby(q,sort=False).transform("size").to_numpy()
    weight/=weight.mean()
    model=LGBMRegressor(objective="regression",n_estimators=350,learning_rate=.03,num_leaves=15,
                         min_child_samples=30,subsample=.8,subsample_freq=1,colsample_bytree=.5,
                         reg_lambda=5,random_state=1941,n_jobs=2,deterministic=True,force_col_wise=True,verbosity=-1)
    model.fit(x,target,sample_weight=weight)
    return model.predict(held)

def five_source_gain(frame, training, held, labels, static, signals, bge, stage2, trees, gain, normalize):
    names = ("giga", "qwen", "qwen_broad", "ru", "jina")
    design = []
    for idx, table in zip((training, held), static):
        part = frame.iloc[idx].reset_index(drop=True)
        design.append(stage2(table.reset_index(drop=True), part, [signals[name][idx] for name in names], None).astype(np.float32))
    train = frame.iloc[training]
    qa = frame.query_id.iloc[held]
    lhs, rhs = trees(train, labels[training], *design)
    reg = gain(train, labels[training], *design)
    pair = .85*normalize(.5*(normalize(lhs,qa)+normalize(rhs,qa)),qa)+.15*bge
    expected = .85*normalize(reg,qa)+.15*bge
    return normalize(.75*normalize(pair,qa)+.25*normalize(expected,qa),qa)

def gain_seed_scores(frame,training,held,labels,static,signals,bge,stage2,normalize,tuning):
    names=("giga","qwen","qwen_broad","ru","jina")
    matrices=[]
    for rows,tab in zip((training,held),static):
        matrix=stage2(tab.reset_index(drop=True),frame.iloc[rows].reset_index(drop=True),[signals[name][rows] for name in names],None)
        matrices.append(matrix.astype(np.float32))
    x,v=matrices
    tr=frame.iloc[training];ev=frame.query_id.iloc[held]
    groups=tr.query_id.to_numpy(str);order=np.lexsort((tr.id.to_numpy(str),groups))
    counts=pd.Series(groups[order]).groupby(groups[order],sort=False).size().tolist()
    yy=np.asarray(labels)[training]
    q=tr.query_id.reset_index(drop=True)
    npos=pd.Series(yy).groupby(q,sort=False).transform("sum").to_numpy(int)
    ideal=np.r_[0.,np.cumsum(1/np.log2(np.arange(2,12)))][np.minimum(npos,10)]
    target=np.divide(yy,ideal,out=np.zeros(len(yy)),where=ideal>0)
    w=1./q.groupby(q,sort=False).transform("size").to_numpy();w/=w.mean()
    sorted_x=x.iloc[order]
    ensemble=[]
    for seed in (805,1907,3031):
        params=dict(tuning,n_jobs=1,random_state=seed,bagging_seed=seed+1,feature_fraction_seed=seed+2)
        a=LGBMRanker(objective="lambdarank",lambdarank_truncation_level=20,**params).fit(sorted_x,yy[order],group=counts).predict(v)
        b=LGBMRanker(objective="rank_xendcg",objective_seed=seed+3,**params).fit(sorted_x,yy[order],group=counts).predict(v)
        g=LGBMRegressor(objective="regression",**params).fit(x,target,sample_weight=w).predict(v)
        pair=.85*normalize(.5*(normalize(a,ev)+normalize(b,ev)),ev)+.15*bge
        gain=.85*normalize(g,ev)+.15*bge
        ensemble.append(normalize(.75*normalize(pair,ev)+.25*normalize(gain,ev),ev))
    return np.stack(ensemble)

In [15]:
"""Prediction functions embedded in the minimal full-list gain notebook."""
def light_sha(path):
    with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()


def light_frozen_folder(name):
    spec=EXTRA_MANIFEST[name];local=os.getenv('RANK_'+name.upper());files=spec['files']
    path=Path(local) if local else Path(snapshot_download(spec['repo'],revision=spec['revision'],allow_patterns=[f['rfilename'] for f in files]))
    if not all(verified(path/f['rfilename'],f) for f in files):raise ValueError('Unrecognized pinned '+name+' model')
    return path


def light_ru(frame,root):
    root.mkdir(exist_ok=True,parents=True);ids=frame.id.to_numpy(str);key=_bge_frame_hash(frame,False);path=root/'scores.npz'
    scores=np.full(len(ids),np.nan,np.float32)
    if path.is_file():
        with np.load(path,allow_pickle=False) as z:
            if not np.array_equal(z['ids'],ids) or z['frame'].item()!=key:raise ValueError('RuModernBERT resume IDs differ')
            scores[:]=z['scores']
    def save():_bge_atomic(path,lambda f:np.savez(f,ids=ids,frame=key,scores=scores))
    if not np.isfinite(scores).all():
        folder=light_frozen_folder('ru');tok=AutoTokenizer.from_pretrained(folder,local_files_only=True,trust_remote_code=False)
        with torch.inference_mode():score_ru(frame,scores,save,folder,tok,'cuda',False)
        gc.collect();torch.cuda.empty_cache()
    if not np.isfinite(scores).all():raise ValueError('Incomplete Russian reranker predictions')
    return scores


def light_base_inputs(notebook,frame,train,test,blind,root,signature):
    """Use a supplied base-score cache when available; otherwise calculate scores."""
    ids=frame.id.to_numpy(str)
    replay=os.getenv('RANK_REPLAY')
    if replay:
        replay_path=Path(replay).expanduser().resolve(strict=True)
        if light_sha(replay_path) not in LIGHT_BASE_SHA:
            raise ValueError('RANK_REPLAY: неизвестный или неподходящий архив базовых оценок')
        with np.load(replay_path,allow_pickle=False) as z:
            if z['data'].item()!=DATA_SHA or not np.array_equal(ids,z['ids']):
                raise ValueError('RANK_REPLAY: оценки относятся к другому датасету или порядку строк')
            signals={k:z[k] for k in ('giga','qwen','qwen_broad','ru')}
            bge_test=z['bge']
        if np.shape(bge_test)!=(len(test),) or not np.isfinite(bge_test).all():
            raise ValueError('RANK_REPLAY: некорректные BGE test-оценки')
        if any(np.shape(v)!=(len(frame),) or not np.isfinite(v).all() for v in signals.values()):
            raise ValueError('RANK_REPLAY: некорректные базовые оценки')
        print('Используется кэш базовых оценок:',replay_path,flush=True)
        return signals,bge_test,None

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Расчёт без базового кэша требует BF16 CUDA GPU. Задай RANK_REPLAY, если есть архив оценок.')

    inner,outer,_=night_partition(train.assign(relevance=0))
    bge_oof=np.full(len(train),np.nan,np.float32)
    outer_predictions=[]
    bge_test_folds=[]
    for k,held in enumerate(inner):
        training=np.sort(np.concatenate([ix for j,ix in enumerate(inner) if j!=k]))
        titles=fit_title_map(train.iloc[training])
        validation=train.iloc[np.r_[held,outer]].drop(columns='relevance').reset_index(drop=True)
        measured=run_bge_fold(
            fold=k,
            train_frame=prepare_frame(train.iloc[training],titles),
            valid_frame=prepare_frame(validation,titles),
            test_frame=prepare_frame(test,titles),
            model_dir=night_model_path('bge',notebook),
            artifact_dir=root/'bge',
            data_signature=DATA_SHA,
            code_signature=signature,
            stop_requested=lambda:False,
            on_progress=lambda event:print(event,flush=True),
        )
        if measured['status']!='complete':
            raise RuntimeError(f'BGE fold {k} не завершён: {measured.get("run_dir")}')
        valid_norm=normalize_by_query(measured['valid_scores'],validation.query_id)
        bge_oof[held]=valid_norm[:len(held)]
        outer_predictions.append(valid_norm[len(held):])
        bge_test_folds.append(normalize_by_query(measured['test_scores'],blind.query_id))
        print(f'BGE fold {k+1}/4 завершён',flush=True)

    bge_oof[outer]=np.mean(np.vstack(outer_predictions),axis=0)
    bge_test=np.mean(np.vstack(bge_test_folds),axis=0)
    if not np.isfinite(bge_oof).all() or not np.isfinite(bge_test).all():
        raise ValueError('Не удалось получить полные OOF/test оценки BGE')

    signals={'giga':night_giga_scores(frame,night_model_path('giga',notebook),root/'giga',signature,lambda:False)}
    for name in ('qwen','qwen_broad'):
        signals[name]=night_qwen_scores(frame,night_model_path('qwen',notebook),root/name,signature,lambda:False,broad=name=='qwen_broad')
    signals['ru']=light_ru(frame,root/'ru')
    if any(np.shape(v)!=(len(frame),) or not np.isfinite(v).all() for v in signals.values()):
        raise ValueError('Неполные или некорректные оценки базовых моделей')
    return signals,bge_test,bge_oof


def light_direction(frame,root,reverse):
    name='RANK_JINA_REVERSE_FULL' if reverse else 'RANK_JINA_FULL_LIST';expected=LIGHT_REVERSE_SHA if reverse else LIGHT_DIRECT_SHA
    given=os.getenv(name);ids=frame.id.to_numpy(str);stage=root/('jina-full-reverse-passages' if reverse else 'jina-full-passages')
    marker_file=stage/'ready.json'
    if given:
        path=Path(given).expanduser().resolve(strict=True)
        if light_sha(path)!=expected:raise ValueError('Unrecognized '+name+' raw measurements')
    elif marker_file.is_file():
        marker=json.loads(marker_file.read_text('utf8'));path=Path(marker['file'])
        if marker['data']!=DATA_SHA or marker['revision']!=LIGHT_REV or marker['rows']!=len(frame) or light_sha(path)!=marker['sha256']:
            raise ValueError('Invalid local full-list measurement')
    else:
        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():raise RuntimeError('Full-list Jina requires BF16 CUDA or the matching signed measurements')
        stage.mkdir(exist_ok=True,parents=True);folder=light_frozen_folder('jina')
        seq,rows,identical,tail=jina_full_plan(frame,folder,reverse=reverse);scores=np.full(len(ids),np.nan,np.float32)
        order='reversed-hash-order-full-v1' if reverse else 'original-hash-order-full-v1'
        key=hashlib.sha256((LIGHT_REV+'-'+order+'-cuda').encode()+ids.tobytes()).hexdigest();path=stage/'scores.npz'
        if path.is_file():
            with np.load(path,allow_pickle=False) as z:
                if z['data'].item()!=DATA_SHA or z['key'].item()!=key or not np.array_equal(z['ids'],ids) or not np.array_equal(z['tail'],tail):raise ValueError('Incompatible Jina resume file')
                scores[:]=z['scores']
        def store():_bge_atomic(path,lambda f:np.savez_compressed(f,ids=ids,scores=scores,tail=tail,key=key,data=DATA_SHA))
        model=jina_full_load(folder,'cuda');last=time.monotonic()
        for i,members in enumerate(rows):
            if np.isfinite(scores[members]).all():continue
            if (root/'STOP').exists():store();raise RuntimeError('Full Jina paused')
            if not reverse and identical[i]:scores[members]=light_short_identical(model,seq[i],len(members),'cuda')
            else:scores[members]=jina_full_score(model,seq[i],len(members),'cuda')
            if time.monotonic()-last>30:
                store();print(name,int(np.isfinite(scores).sum()),'/',len(frame),flush=True);last=time.monotonic()
        store();del model;gc.collect();torch.cuda.empty_cache()
        if not np.isfinite(scores).all():raise ValueError('Incomplete full Jina run')
        record=dict(data=DATA_SHA,revision=LIGHT_REV,rows=len(frame),file=str(path.resolve()),sha256=light_sha(path))
        _bge_atomic(marker_file,lambda f:f.write(json.dumps(record).encode()))
    with np.load(path,allow_pickle=False) as z:
        if z['data'].item()!=DATA_SHA or not np.array_equal(z['ids'],ids):raise ValueError('Misaligned '+name+' row IDs')
        values=z['scores']
    if values.shape!=(len(frame),) or not np.isfinite(values).all():raise ValueError('Incomplete full Jina logits')
    return values


def make_submission(notebook,destination):
    train,test,sample,info=load_competition_data(night_data_path(notebook))
    if info['dataset_sha256']!=DATA_SHA:raise ValueError('Unexpected competition data')
    frame=prepare_frame(pd.concat([train.drop(columns='relevance'),test],ignore_index=True),{});n=len(train)
    training=np.arange(n);held=np.arange(n,len(frame));blind=frame.iloc[held].reset_index(drop=True)
    destination.mkdir(exist_ok=True,parents=True);signature=light_sha(notebook);root=destination/('cache-'+signature[:12]+'-'+DATA_SHA[:12]);root.mkdir(exist_ok=True)
    with FileLock(root/'run.lock',timeout=0):
        signals,bge,_bge_oof=light_base_inputs(notebook,frame,train,test,blind,root,signature)
        forward=light_direction(frame,root,False);backward=light_direction(frame,root,True);signals['jina']=.5*(forward+backward)
        # Historical full-data SVD used three BLAS threads, including for the final test rows.
        with threadpool_limits(limits=3):
            context=historical_fit(frame.iloc[training]);static=[historical_transform(frame.iloc[ix].reset_index(drop=True),context) for ix in (training,held)]
        with threadpool_limits(limits=1):
            pred=five_source_gain(frame,training,held,train.relevance.to_numpy(),static,signals,bge,historical_stage2,tree_scores,standalone_gain,normalize_by_query)
            additional=gain_seed_scores(frame,training,held,train.relevance.to_numpy(),static,signals,bge,historical_stage2,normalize_by_query,TUNING)
            pred=np.mean(np.vstack((pred,additional)),axis=0)
        sub=build_submission(sample,test,pred);raw=sub.to_csv(index=False,float_format='%.12g').encode()
        parsed=pd.read_csv(io.BytesIO(raw),dtype={'id':str})
        if not parsed.id.equals(sample.id) or not np.allclose(parsed.prediction,sub.prediction,rtol=1e-10,atol=1e-12):raise ValueError('Submission failed row-order or precision check')
        out=destination/'submission.csv';_bge_atomic(out,lambda f:f.write(raw));print('Сохранено',len(sub),'строк.',out.resolve(),'SHA256',hashlib.sha256(raw).hexdigest())
    return out


In [16]:
warnings.filterwarnings("ignore",message=".*match groups.*",category=UserWarning)
notebook=Path(os.getenv("RANK_NOTEBOOK",str(Path.cwd()/"solution_hybrid_runtime.ipynb"))).resolve()
if not notebook.exists():
    notebook.write_text("Hybrid ranking notebook runtime marker.",encoding="utf-8")
destination=Path(os.getenv("RANK_OUTPUT",str(Path.cwd()/"ranking_output"))).resolve()
destination.mkdir(exist_ok=True,parents=True)

train,test,sample,info=load_competition_data(night_data_path(notebook))
if info['dataset_sha256']!=DATA_SHA:
    raise ValueError('Загруженный ZIP не совпадает с ожидаемым датасетом соревнования.')
frame=prepare_frame(pd.concat([train.drop(columns='relevance'),test],ignore_index=True),{})
n_train=len(train)
blind=frame.iloc[n_train:].reset_index(drop=True)
signature=light_sha(notebook)
root=destination/('hybrid-cache-'+signature[:12]+'-'+DATA_SHA[:12])
root.mkdir(exist_ok=True,parents=True)

with FileLock(root/'run.lock',timeout=0):
    signals,bge_test,bge_oof=light_base_inputs(notebook,frame,train,test,blind,root,signature)
    forward=light_direction(frame,root,False)
    backward=light_direction(frame,root,True)
    signals['jina']=.5*(forward+backward)

folds,groups=build_group_folds(train,n_folds=5)
labels=train.relevance.to_numpy()
oof_pred=np.full(n_train,np.nan,dtype=np.float64)
fold_rows=[]
use_bge_oof = bge_oof is not None
for fold_no,(tr_idx,va_idx) in enumerate(folds,start=1):
    fold_context=historical_fit(frame.iloc[tr_idx])
    fold_static=[historical_transform(frame.iloc[ix].reset_index(drop=True),fold_context) for ix in (tr_idx,va_idx)]
    fold_bge = bge_oof[va_idx] if use_bge_oof else np.zeros(len(va_idx),dtype=np.float32)
    base_pred=five_source_gain(
        frame,tr_idx,va_idx,labels,fold_static,signals,fold_bge,
        historical_stage2,tree_scores,standalone_gain,normalize_by_query
    )
    seed_pred=gain_seed_scores(
        frame,tr_idx,va_idx,labels,fold_static,signals,fold_bge,
        historical_stage2,normalize_by_query,TUNING
    )
    fold_pred=np.mean(np.vstack((base_pred,seed_pred)),axis=0)
    oof_pred[va_idx]=fold_pred
    fold_score=mean_ndcg(train.iloc[va_idx].reset_index(drop=True),fold_pred,k=10)
    fold_rows.append({'fold':fold_no,'ndcg@10':fold_score,
                      'validation_queries':int(train.iloc[va_idx].query_id.nunique()),
                      'validation_rows':int(len(va_idx))})
    print(f'Fold {fold_no}/5 — NDCG@10: {fold_score:.6f}',flush=True)

if not np.isfinite(oof_pred).all():
    raise ValueError('OOF-предсказания неполные')
cv_score=mean_ndcg(train.reset_index(drop=True),oof_pred,k=10)
metrics={
    'metric':'NDCG@10',
    'oof_ndcg_at_10':cv_score,
    'folds':fold_rows,
    'bge_included':bool(use_bge_oof),
    'note':('Grouped 5-fold OOF; BGE train scores generated out-of-fold.'
            if use_bge_oof else
            'Grouped 5-fold OOF without BGE: supplied replay cache contains BGE test scores but no BGE OOF train scores.')
}
metrics_path=destination/'validation_metrics.json'
metrics_path.write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'\nИтоговый OOF NDCG@10 (5 folds): {cv_score:.6f}')
print('Метрики сохранены:',metrics_path)

training=np.arange(n_train)
held=np.arange(n_train,len(frame))
with threadpool_limits(limits=3):
    context=historical_fit(frame.iloc[training])
    static=[historical_transform(frame.iloc[ix].reset_index(drop=True),context) for ix in (training,held)]
with threadpool_limits(limits=1):
    pred=five_source_gain(frame,training,held,labels,static,signals,bge_test,
                          historical_stage2,tree_scores,standalone_gain,normalize_by_query)
    additional=gain_seed_scores(frame,training,held,labels,static,signals,bge_test,
                                historical_stage2,normalize_by_query,TUNING)
    pred=np.mean(np.vstack((pred,additional)),axis=0)
sub=build_submission(sample,test,pred)
raw=sub.to_csv(index=False,float_format='%.12g').encode()
parsed=pd.read_csv(io.BytesIO(raw),dtype={'id':str})
if not parsed.id.equals(sample.id) or not np.allclose(parsed.prediction,sub.prediction,rtol=1e-10,atol=1e-12):
    raise ValueError('Проверка submission.csv не пройдена')
out=destination/'submission.csv'
_bge_atomic(out,lambda f:f.write(raw))
print('Сохранено:',out)


Используется кэш базовых оценок: /content/RANK_REPLAY/base-features.npz
Fold 1/5 — NDCG@10: 0.974262
Fold 2/5 — NDCG@10: 0.975063
Fold 3/5 — NDCG@10: 0.964398
Fold 4/5 — NDCG@10: 0.964785
Fold 5/5 — NDCG@10: 0.967290

Итоговый OOF NDCG@10 (5 folds): 0.969153
Метрики сохранены: /content/ranking_output/validation_metrics.json
Сохранено: /content/ranking_output/submission.csv
